# CDG-DFD Phase 2, E1c: external test on unseen generators (ArtiFact faces)

Test only, no training, **no downloads**. The E1 and E1b checkpoints are scored on the face images of **ArtiFact**
(Rahman et al., ICIP 2023), which is already hosted on Kaggle. None of its generators was used in training
(training saw only Stable Diffusion 1.5 inpainting and text-to-image from DFF).

- Reals: the FFHQ and CelebA-HQ folders. Fakes: the face subsets of the other folders: StyleGAN 1/2/3, CIPS,
  GANformer, ProjectedGAN and Taming Transformer (their FFHQ models), Diffusion-StyleGAN2 (FFHQ), denoising-diffusion
  GAN, LaMa (inpainted CelebA-HQ), StarGAN, SFHQ, FaceSynthetics, and Stable Diffusion's Male/Female prompts.
  The first table printed lists every folder, what it holds and how many images were taken.
- ArtiFact put **every** image (real and fake) through the same random crop, downscale and JPEG compression and
  stored it at 200x200, so the compression shortcut we found in DFF is absent here by construction.
- Faces are cropped exactly as in Phase 1 (MTCNN, 224, margin 40), in two versions: `asis` and `matched`
  (both classes re-saved as 512x512 JPEG q75 first).

**Inputs (Add Input):** the dataset **awsaf49/artifact-dataset** (search "ArtiFact"), plus the outputs of
**oct-22** (E1) and **oct-23-1** (E1b) for the checkpoints. **Settings:** GPU T4 (one is enough), Internet on
(only for one small pip install). **Time:** about 1 hour.

**Please run it with "Run All" in the open editor, not "Save Version"**: a saved version that fails shows no output
at all, which is why we could not see the last two failures. If a cell fails, the error stays visible under it.

In [ ]:
!pip install -q facenet-pytorch --no-deps
import os, glob, json, subprocess, shutil
import pandas as pd
import torch
from IPython.display import display, Markdown
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
OUT = '/kaggle/working/external'
os.makedirs(OUT, exist_ok=True)

def sh(cmd):
    # runs a step, shows its output live and also keeps it in external/log.txt
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    with open(f'{OUT}/log.txt', 'a') as logf:
        for line in p.stdout:
            if 'Warning' in line:
                continue
            print(line, end='', flush=True); logf.write(line)
            tail = (tail + [line])[-30:]
    if p.wait() != 0:
        raise RuntimeError(f'failed (exit {p.returncode}):' + chr(10) + ''.join(tail))

In [ ]:
%%writefile cdg_phase1.py
#!/usr/bin/env python3
"""
CDG-DFD Phase 1: data and protocol rebuild.

Answers Reviewer 2 Comments 2, 3, 7, 10 and 12 by rebuilding every split from
the raw files, so each number in the paper can be traced to code.

Stages (run in order; every stage reads and writes under --out):

  scan            index raw files / sampled video frames      -> raw_index.csv
  crop            MTCNN crops, lossless PNG, resumable         -> crop_status.csv, crops/
  dedup           exact + near-duplicate detection             -> dedup.csv, near_dup_pairs.csv
  split           group-disjoint, leave-one-generator-out      -> splits/fold_*/{train,val,test}.csv
  audit           leakage checks, accounting + domain tables   -> reports/*.md, reports/*.csv
  audit-original  re-creates the first-edition 85/15 split and
                  measures how much leaked across it          -> reports/audit_original.md

Label conventions (unchanged from the paper):
  y = 1 fake, 0 real
  d = 0 GAN-era family (HardFake, CelebDF-v2), 1 diffusion family (DFF),
      -1 = never used for training (test-only sets; no domain label needed)
"""
from __future__ import annotations

import argparse
import hashlib
import json
import os
import random
import re
import sys
import time
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from PIL import Image

IMG_EXT = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')
VID_EXT = ('.mp4', '.avi', '.mov', '.mkv')

# ----------------------------------------------------------------------------
# Source definitions
# ----------------------------------------------------------------------------
# family: real | gan | faceswap_ae | diffusion | faceswap
SOURCES = {
    'hardfake_real':  dict(dataset='HardFake', generator='real',          family='real',        label=0, domain=0),
    'hardfake_fake':  dict(dataset='HardFake', generator='hardfake_gan',  family='gan',         label=1, domain=0),
    'celebdf_real':   dict(dataset='CelebDF',  generator='real',          family='real',        label=0, domain=0),
    'celebdf_fake':   dict(dataset='CelebDF',  generator='celebdf_swap',  family='faceswap_ae', label=1, domain=0),
    'dff_wiki':       dict(dataset='DFF',      generator='real',          family='real',        label=0, domain=1),
    'dff_inpainting': dict(dataset='DFF',      generator='sd_inpainting', family='diffusion',   label=1, domain=1),
    'dff_text2img':   dict(dataset='DFF',      generator='sd15_text2img', family='diffusion',   label=1, domain=1),
    # InsightFace is a face-swap toolbox, not diffusion: test-only, never labelled d=1.
    'dff_insight':    dict(dataset='DFF',      generator='insightface',   family='faceswap',    label=1, domain=-1),
    # Shortcut diagnostic: the same wiki reals passed through the fakes' file pipeline (512x512, JPEG q75)
    # before cropping. Test-only. If a detector flags these as fake, it is reading compression, not faces.
    'dff_wiki_matched': dict(dataset='DFF',    generator='real_matched',  family='real',        label=0, domain=-1),
}
MATCHED_SIZE, MATCHED_QUALITY = (512, 512), 75   # DFF fakes: all 512x512, IJG quality 75 (PIL's default)
DFF_SUBSETS = {'dff_wiki': 'wiki', 'dff_inpainting': 'inpainting',
               'dff_text2img': 'text2img', 'dff_insight': 'insight'}

# Leave-one-generator-out folds over the two DFF diffusion generators.
FOLDS = {
    'A': dict(train_gens=['sd_inpainting'], heldout_gens=['sd15_text2img']),
    'B': dict(train_gens=['sd15_text2img'], heldout_gens=['sd_inpainting']),
}
# Generators that are only ever tested on (in every fold).
TEST_ONLY_GENS = {'insightface', 'real_matched'}

# Standard IJG luminance quantisation table (quality 50), used to estimate JPEG quality.
_IJG_LUMA = np.array([
    16, 11, 10, 16, 24, 40, 51, 61, 12, 12, 14, 19, 26, 58, 60, 55,
    14, 13, 16, 24, 40, 57, 69, 56, 14, 17, 22, 29, 51, 87, 80, 62,
    18, 22, 37, 56, 68, 109, 103, 77, 24, 35, 55, 64, 81, 104, 113, 92,
    49, 64, 78, 87, 103, 121, 120, 101, 72, 92, 95, 98, 112, 100, 103, 99], dtype=np.float64)


def log(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True)


def uid_of(*parts) -> str:
    return hashlib.sha1('|'.join(map(str, parts)).encode()).hexdigest()[:16]


def md5_file(path, chunk=1 << 20) -> str:
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


def list_files(root, exts):
    out = []
    for dp, _, fns in os.walk(root):
        for fn in fns:
            if fn.lower().endswith(exts):
                out.append(os.path.join(dp, fn))
    return sorted(out)


def estimate_jpeg_quality(im) -> float:
    """Estimate IJG quality from the luminance quantisation table; NaN if not JPEG."""
    q = getattr(im, 'quantization', None)
    if not q or 0 not in q:
        return float('nan')
    table = np.array(q[0], dtype=np.float64)
    if table.size != 64:
        return float('nan')
    scale = float(np.mean(table / _IJG_LUMA) * 100.0)
    qual = (200.0 - scale) / 2.0 if scale <= 100 else 5000.0 / scale
    return round(float(np.clip(qual, 1, 100)), 1)


def image_header(path):
    try:
        with Image.open(path) as im:
            return (im.format or os.path.splitext(path)[1][1:].upper(),
                    im.size[0], im.size[1], estimate_jpeg_quality(im))
    except Exception:
        return ('UNREADABLE', -1, -1, float('nan'))


def find_subdir(root, name):
    """DFF zips extract either to root/name or root/name/name; return the deepest match."""
    cands = [os.path.join(root, name, name), os.path.join(root, name)]
    for c in cands:
        if os.path.isdir(c):
            return c
    return None


# Identity parsing --------------------------------------------------------------
_WIKI_ID = re.compile(r'^(\d+)_')          # IMDB-WIKI wiki files:  10049200_1891-09-16_1958.jpg
_IMDB_ID = re.compile(r'^(nm\d+)_')        # IMDB files:            nm0000100_rm1001569280_1955-1-6_2011.jpg
_CDF_REAL = re.compile(r'^(id\d+)_\d+$')   # Celeb-real/id0_0000
_CDF_FAKE = re.compile(r'^(id\d+)_(id\d+)_\d+$')  # Celeb-synthesis/id0_id16_0000


def dff_identity(basename):
    stem = os.path.basename(basename)
    for rx, tag in ((_WIKI_ID, 'wiki'), (_IMDB_ID, 'imdb')):
        m = rx.match(stem)
        if m:
            return f'{tag}:{m.group(1)}'
    return ''


def celebdf_identities(video_id):
    folder, stem = video_id.split('/', 1) if '/' in video_id else ('', video_id)
    m = _CDF_FAKE.match(stem)
    if m:
        return [f'cdf:{m.group(1)}', f'cdf:{m.group(2)}']
    m = _CDF_REAL.match(stem)
    if m:
        return [f'cdf:{m.group(1)}']
    return []  # YouTube-real videos carry no identity label


# ----------------------------------------------------------------------------
# Stage 1: scan
# ----------------------------------------------------------------------------
def row_template(source, raw_path, **kw):
    meta = SOURCES.get(source, None)
    r = dict(uid='', source=source, raw_path=raw_path, frame_idx=-1, rel_key='', video_id='',
             identities='', celebdf_official_test='', orig_format='', orig_w=-1, orig_h=-1,
             jpeg_q=float('nan'), md5_raw='')
    if meta:
        r.update(meta)
    r.update(kw)
    return r


def scan_images(source, root, rel_root, with_identity):
    rows = []
    for p in list_files(root, IMG_EXT):
        rel = os.path.relpath(p, rel_root)
        rel_key = os.path.splitext(rel)[0].lower().replace('\\', '/')
        fmt, w, h, q = image_header(p)
        ident = dff_identity(p) if with_identity else ''
        rows.append(row_template(source, p, uid=uid_of(source, rel), rel_key=rel_key, identities=ident,
                                 orig_format=fmt, orig_w=w, orig_h=h, jpeg_q=q, md5_raw=md5_file(p)))
    return rows


def read_celebdf_test_list(path):
    vids = set()
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            rel = parts[-1]
            vids.add(os.path.splitext(rel)[0].replace('\\', '/'))
    return vids


def scan_celebdf(root, test_list, frames_real, frames_fake, max_fake_train_videos, seed):
    import cv2
    test_vids = read_celebdf_test_list(test_list)
    log(f'CelebDF official test list: {len(test_vids)} videos')
    vids = []
    for folder, source in (('Celeb-real', 'celebdf_real'), ('YouTube-real', 'celebdf_real'),
                           ('Celeb-synthesis', 'celebdf_fake')):
        d = os.path.join(root, folder)
        if not os.path.isdir(d):
            log(f'  WARNING: {d} not found')
            continue
        for p in list_files(d, VID_EXT):
            vid = f'{folder}/{os.path.splitext(os.path.basename(p))[0]}'
            vids.append((source, p, vid))
    # Optional cap on fake training videos (test videos are always kept).
    if max_fake_train_videos:
        rng = random.Random(seed)
        fake_train = [v for v in vids if v[0] == 'celebdf_fake' and v[2] not in test_vids]
        keep = set(x[2] for x in rng.sample(fake_train, min(max_fake_train_videos, len(fake_train))))
        vids = [v for v in vids if not (v[0] == 'celebdf_fake' and v[2] not in test_vids and v[2] not in keep)]
    missing = test_vids - set(v[2] for v in vids)
    if missing:
        log(f'  WARNING: {len(missing)} official test videos not found on disk (e.g. {sorted(missing)[:3]})')
    rows = []
    for source, p, vid in vids:
        cap = cv2.VideoCapture(p)
        n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap.release()
        k = frames_real if source == 'celebdf_real' else frames_fake
        if n <= 0:
            idxs = [-2]  # unreadable video; recorded so it shows up in the accounting
        else:
            idxs = sorted(set(np.linspace(0, n - 1, num=k + 2)[1:-1].round().astype(int).tolist()))
        for fi in idxs:
            rows.append(row_template(source, p, uid=uid_of(source, vid, fi), frame_idx=fi, video_id=vid,
                                     identities=';'.join(celebdf_identities(vid)),
                                     celebdf_official_test=int(vid in test_vids),
                                     orig_format='video:' + os.path.splitext(p)[1][1:].lower(),
                                     orig_w=w, orig_h=h))
    return rows


def scan_external(name, root):
    """External test set: <root>/real/** and <root>/fake/** (either may be missing)."""
    rows = []
    for sub, label in (('real', 0), ('fake', 1)):
        d = os.path.join(root, sub)
        if not os.path.isdir(d):
            continue
        src = f'ext_{name}_{sub}'
        for p in list_files(d, IMG_EXT):
            rel = os.path.relpath(p, root)
            fmt, w, h, q = image_header(p)
            gen = 'real' if label == 0 else f'ext_{name}'
            rows.append(row_template(src, p, uid=uid_of(src, rel), rel_key=rel.lower(), dataset=f'ext_{name}',
                                     generator=gen, family='real' if label == 0 else 'external',
                                     label=label, domain=-1, orig_format=fmt, orig_w=w, orig_h=h,
                                     jpeg_q=q, md5_raw=md5_file(p)))
    return rows


def matched_rows(wiki_rows):
    out = []
    for x in wiki_rows:
        y = dict(x)
        y.update(SOURCES['dff_wiki_matched'], source='dff_wiki_matched',
                 uid=uid_of('dff_wiki_matched', x['rel_key']),
                 md5_raw='matched:' + str(x['md5_raw']), orig_format='JPEG(matched)',
                 orig_w=MATCHED_SIZE[0], orig_h=MATCHED_SIZE[1], jpeg_q=float(MATCHED_QUALITY))
        out.append(y)
    return out


def stage_add_matched(a):
    """Add the matched-real rows to an existing rebuild (needs the wiki images at their scanned paths).
    Then run crop (only the new rows are cropped), split and audit. dedup need not be re-run: the new
    rows share a person and source photo with their wiki originals, so they always join those groups."""
    path = os.path.join(a.out, 'raw_index.csv')
    df = pd.read_csv(path, keep_default_na=False, low_memory=False)
    if (df.source == 'dff_wiki_matched').any():
        log('matched rows already present; nothing to do')
        return
    wiki = df[df.source == 'dff_wiki']
    missing = (~wiki.raw_path.map(os.path.exists)).sum()
    if missing:
        sys.exit(f'{missing} of {len(wiki)} wiki images are not at their scanned paths; re-download wiki.zip first')
    m = pd.DataFrame(matched_rows(wiki.to_dict('records')))
    pd.concat([df, m[df.columns]], ignore_index=True).to_csv(path, index=False)
    log(f'added {len(m)} dff_wiki_matched rows; raw_index.csv now {len(df) + len(m)} rows')


def stage_scan(a):
    os.makedirs(a.out, exist_ok=True)
    rows = []
    if a.hardfake_root:
        for sub, src in (('real', 'hardfake_real'), ('fake', 'hardfake_fake')):
            d = os.path.join(a.hardfake_root, sub)
            r = scan_images(src, d, d, with_identity=False)
            log(f'{src}: {len(r)} images from {d}')
            rows += r
    if a.dff_root:
        for src, sub in DFF_SUBSETS.items():
            d = find_subdir(a.dff_root, sub)
            if d is None:
                log(f'{src}: not found under {a.dff_root} (skipped)')
                continue
            r = scan_images(src, d, d, with_identity=True)
            if src == 'dff_insight' and a.dff_insight_sample and len(r) > a.dff_insight_sample:
                r = random.Random(a.seed).sample(r, a.dff_insight_sample)  # test-only set: a sample is enough
            log(f'{src}: {len(r)} images from {d}')
            rows += r
            if src == 'dff_wiki' and a.dff_matched_real:
                m = matched_rows(r)
                log(f'dff_wiki_matched: {len(m)} pipeline-matched copies of the wiki reals (test-only)')
                rows += m
    if a.celebdf_root:
        tl = a.celebdf_test_list or os.path.join(a.celebdf_root, 'List_of_testing_videos.txt')
        r = scan_celebdf(a.celebdf_root, tl, a.celebdf_frames_real, a.celebdf_frames_fake,
                         a.celebdf_max_fake_train_videos, a.seed)
        log(f'celebdf: {len(r)} frames from {len(set(x["video_id"] for x in r))} videos')
        rows += r
    for spec in a.external or []:
        name, root = spec.split('=', 1)
        r = scan_external(name, root)
        log(f'external {name}: {len(r)} images')
        rows += r
    if not rows:
        sys.exit('Nothing scanned: give at least one of --hardfake-root/--dff-root/--celebdf-root/--external')
    df = pd.DataFrame(rows)
    dupe_uid = df.uid.duplicated()
    if dupe_uid.any():
        sys.exit(f'uid collision on {dupe_uid.sum()} rows; check input paths')
    df.to_csv(os.path.join(a.out, 'raw_index.csv'), index=False)
    cfg = {k: v for k, v in vars(a).items() if k != 'func'}
    with open(os.path.join(a.out, 'config.json'), 'w') as f:
        json.dump(cfg, f, indent=2)
    log(f'raw_index.csv: {len(df)} rows')
    if 'dff_wiki' in set(df.source):
        wiki_keys = set(df.loc[df.source == 'dff_wiki', 'rel_key'])
        for src in ('dff_inpainting', 'dff_text2img', 'dff_insight'):
            sub = df[df.source == src]
            if len(sub):
                log(f'  {src}: {sub.rel_key.isin(wiki_keys).mean()*100:.1f}% share a relative path with a wiki '
                    f'real image; identity parsed for {(sub.identities != "").mean()*100:.1f}%')


# ----------------------------------------------------------------------------
# Stage 2: crop (MTCNN, identical settings to the first edition)
# ----------------------------------------------------------------------------
def match_fake_pipeline(img):
    """Resize to 512x512 and JPEG-encode at quality 75, as every DFF fake file is stored."""
    import io
    buf = io.BytesIO()
    img.resize(MATCHED_SIZE, Image.Resampling.BICUBIC).save(buf, format='JPEG', quality=MATCHED_QUALITY)
    buf.seek(0)
    return Image.open(buf).convert('RGB')


def stage_crop(a):
    import torch
    from facenet_pytorch import MTCNN
    df = pd.read_csv(os.path.join(a.out, 'raw_index.csv'), keep_default_na=False, low_memory=False)
    status_path = os.path.join(a.out, 'crop_status.csv')
    done = set()
    if os.path.exists(status_path):
        done = set(pd.read_csv(status_path, usecols=['uid']).uid)
    todo = df[~df.uid.isin(done)]
    if a.limit:
        todo = todo.head(a.limit)
    log(f'crop: {len(done)} already done, {len(todo)} to do')
    if todo.empty:
        return
    device = a.device or ('cuda' if torch.cuda.is_available() else 'cpu')
    # Same settings as df-1-first-edition: 224 px, 40 px margin, no post-processing, largest face.
    mtcnn = MTCNN(image_size=224, margin=40, post_process=False, keep_all=False, device=device)
    crop_root = os.path.join(a.out, 'crops')
    header = not os.path.exists(status_path)
    buf, n_ok, n_fail, t0 = [], 0, 0, time.time()

    def flush():
        nonlocal buf, header
        if buf:
            pd.DataFrame(buf).to_csv(status_path, mode='a', header=header, index=False)
            header, buf = False, []

    def handle(row, pil_img, err=''):
        nonlocal n_ok, n_fail
        out_dir = os.path.join(crop_root, row.source)
        os.makedirs(out_dir, exist_ok=True)
        out_path = os.path.join(out_dir, f'{row.uid}.png')
        ok, prob = 0, float('nan')
        if pil_img is not None:
            try:
                face, prob = mtcnn(pil_img, return_prob=True)
                if face is not None:
                    arr = face.permute(1, 2, 0).clamp(0, 255).byte().cpu().numpy()
                    Image.fromarray(arr).save(out_path)  # PNG: no extra lossy step on either class
                    ok = 1
                else:
                    err = 'no_face'
            except Exception as e:  # noqa: BLE001
                err = f'mtcnn_error:{type(e).__name__}'
        crop_md5 = md5_file(out_path) if ok else ''
        buf.append(dict(uid=row.uid, mtcnn_ok=ok, mtcnn_prob=prob if prob is not None else float('nan'),
                        crop_path=os.path.relpath(out_path, a.out) if ok else '', crop_md5=crop_md5,
                        fail_reason=err))
        n_ok += ok
        n_fail += 1 - ok
        if (n_ok + n_fail) % 2000 == 0:
            flush()
            rate = (n_ok + n_fail) / max(time.time() - t0, 1e-6)
            log(f'  {n_ok + n_fail}/{len(todo)}  ok={n_ok} fail={n_fail}  {rate:.1f} img/s')

    img_rows = todo[todo.frame_idx.astype(int) == -1]
    vid_rows = todo[todo.frame_idx.astype(int) != -1]
    for row in img_rows.itertuples(index=False):
        try:
            img = Image.open(row.raw_path).convert('RGB')
            if row.source == 'dff_wiki_matched':
                img = match_fake_pipeline(img)
            handle(row, img)
        except Exception as e:  # noqa: BLE001
            handle(row, None, f'read_error:{type(e).__name__}')
    if len(vid_rows):
        import cv2
        for path, grp in vid_rows.groupby('raw_path'):
            cap = cv2.VideoCapture(path)
            for row in grp.sort_values('frame_idx').itertuples(index=False):
                fi = int(row.frame_idx)
                if fi < 0:
                    handle(row, None, 'video_unreadable')
                    continue
                cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
                ok, frame = cap.read()
                if not ok:
                    handle(row, None, 'frame_read_error')
                    continue
                handle(row, Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
            cap.release()
    flush()
    log(f'crop done: ok={n_ok} fail={n_fail}')


# ----------------------------------------------------------------------------
# Stage 3: dedup
# ----------------------------------------------------------------------------
def image_signature(path):
    """64-bit dHash (for candidate search) and a 32x32 grayscale thumbnail (for confirmation).

    Computed on the whole raw image for stills: two MTCNN crops of the same photo can be offset by a
    few pixels, which hides a re-encoded or resized copy, while the full-image thumbnails still match."""
    with Image.open(path) as im:
        g = im.convert('L')
        d = np.asarray(g.resize((9, 8), Image.Resampling.LANCZOS), dtype=np.int16)
        thumb = np.asarray(g.resize((32, 32), Image.Resampling.BOX), dtype=np.uint8)
    bits = (d[:, 1:] > d[:, :-1]).flatten()
    return int(''.join('1' if b else '0' for b in bits), 2), thumb


def near_dup_pairs(hashes, max_dist):
    """All index pairs with Hamming distance <= max_dist (pigeonhole over max_dist+1 chunks)."""
    n_chunks = max_dist + 1
    edges = [int(round(x)) for x in np.linspace(0, 64, n_chunks + 1)]
    pairs = set()
    for c in range(n_chunks):
        lo, hi = edges[c], edges[c + 1]
        mask = (1 << (hi - lo)) - 1
        buckets = defaultdict(list)
        for i, h in enumerate(hashes):
            buckets[(h >> lo) & mask].append(i)
        for members in buckets.values():
            if len(members) < 2:
                continue
            for x in range(len(members)):
                hx = hashes[members[x]]
                for y in range(x + 1, len(members)):
                    i, j = members[x], members[y]
                    if (hx ^ hashes[j]).bit_count() <= max_dist:
                        pairs.add((i, j) if i < j else (j, i))
    return sorted(pairs)


def load_merged(out):
    raw = pd.read_csv(os.path.join(out, 'raw_index.csv'), keep_default_na=False, low_memory=False)
    st = pd.read_csv(os.path.join(out, 'crop_status.csv'), keep_default_na=False)
    st = st.drop_duplicates('uid', keep='last')
    df = raw.merge(st, on='uid', how='left')
    df['mtcnn_ok'] = pd.to_numeric(df['mtcnn_ok'], errors='coerce').fillna(-1).astype(int)  # -1 = not cropped yet
    for c in ('label', 'domain', 'frame_idx'):
        df[c] = df[c].astype(int)
    return df


def stage_dedup(a):
    df = load_merged(a.out)
    if (df.mtcnn_ok == -1).any():
        sys.exit(f'{(df.mtcnn_ok == -1).sum()} rows not cropped yet; finish the crop stage first')
    ok = df[df.mtcnn_ok == 1].copy()
    log(f'dedup: hashing {len(ok)} crops')
    # stills: signature of the raw image; video frames (raw not stored): signature of the crop.
    # Signatures are cached so dedup can be re-run after the raw downloads are deleted.
    cache_path = os.path.join(a.out, 'signatures.npz')
    cache = {}
    if os.path.exists(cache_path):
        z = np.load(cache_path, allow_pickle=False)
        cache = {u: (int(h, 16), t) for u, h, t in zip(z['uid'], z['dhash'], z['thumb'])}
    sigs, missing = [], 0
    for r in ok.itertuples(index=False):
        if r.uid in cache:
            sigs.append(cache[r.uid])
            continue
        src = r.raw_path if r.frame_idx == -1 else os.path.join(a.out, r.crop_path)
        if r.frame_idx == -1 and not os.path.exists(src):
            missing += 1
            src = os.path.join(a.out, r.crop_path)   # weaker fallback: crop-based signature
        sigs.append(image_signature(src))
    if missing:
        log(f'  WARNING: {missing} raw files missing and not cached; used crop-based signatures for them')
    np.savez(cache_path, uid=ok.uid.to_numpy(dtype=str), dhash=np.array([f'{h:016x}' for h, _ in sigs]),
             thumb=np.stack([t for _, t in sigs]))
    ok['dhash'] = pd.Series([h for h, _ in sigs], index=ok.index, dtype=object)
    ok['_thumb'] = pd.Series([t for _, t in sigs], index=ok.index, dtype=object)
    # Exact duplicates: identical raw file (images) or identical crop file. Keep the first by a fixed order.
    order = list(SOURCES) + sorted(set(ok.source) - set(SOURCES))
    ok['_ord'] = ok.source.map({s: i for i, s in enumerate(order)})
    ok = ok.sort_values(['_ord', 'raw_path', 'frame_idx']).reset_index(drop=True)
    first_raw, first_crop, dup_of = {}, {}, []
    for r in ok.itertuples(index=False):
        d = ''
        if r.md5_raw and r.md5_raw in first_raw:
            d = first_raw[r.md5_raw]
        elif r.crop_md5 in first_crop:
            d = first_crop[r.crop_md5]
        else:
            if r.md5_raw:
                first_raw[r.md5_raw] = r.uid
            first_crop[r.crop_md5] = r.uid
        dup_of.append(d)
    ok['exact_dup_of'] = dup_of
    keep = ok[ok.exact_dup_of == ''].reset_index(drop=True)
    log(f'  exact duplicates removed: {(ok.exact_dup_of != "").sum()}')
    hashes = [int(h) for h in keep.dhash.tolist()]
    thumbs = keep._thumb.tolist()
    cand = near_dup_pairs(hashes, a.near_dup_hamming)
    # Unrelated images can share a 64-bit dHash, so every candidate is confirmed by pixel distance on
    # 32x32 thumbnails. Only near-identical images (re-encodes, resizes, small edits) survive.
    rows = []
    for i, j in cand:
        mad = float(np.abs(thumbs[i].astype(np.int16) - thumbs[j].astype(np.int16)).mean())
        if mad <= a.near_dup_mad:
            rows.append((keep.uid[i], keep.uid[j], keep.source[i], keep.source[j],
                         (hashes[i] ^ hashes[j]).bit_count(), round(mad, 2)))
    pr = pd.DataFrame(rows, columns=['uid_a', 'uid_b', 'source_a', 'source_b', 'hamming', 'thumb_mad'])
    pr.to_csv(os.path.join(a.out, 'near_dup_pairs.csv'), index=False)
    log(f'  near-duplicate candidates (dHash Hamming <= {a.near_dup_hamming}): {len(cand)}; '
        f'confirmed (32x32 mean abs diff <= {a.near_dup_mad}): {len(pr)}')
    if len(pr):
        log('  confirmed pairs by source: ' + ', '.join(
            f'{k[0]}~{k[1]}: {v}' for k, v in Counter(zip(pr.source_a, pr.source_b)).most_common(8)))
    ok['dhash'] = ok.dhash.map(lambda h: f'{h:016x}')
    ok[['uid', 'dhash', 'exact_dup_of']].to_csv(os.path.join(a.out, 'dedup.csv'), index=False)


# ----------------------------------------------------------------------------
# Stage 4: split
# ----------------------------------------------------------------------------
class UnionFind:
    def __init__(self):
        self.p = {}

    def find(self, x):
        p = self.p
        p.setdefault(x, x)
        root = x
        while p[root] != root:
            root = p[root]
        while p[x] != root:
            p[x], x = root, p[x]
        return root

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            if ra < rb:
                ra, rb = rb, ra
            self.p[ra] = rb


def load_eligible(out):
    df = load_merged(out)
    dd = pd.read_csv(os.path.join(out, 'dedup.csv'), keep_default_na=False)
    df = df.merge(dd, on='uid', how='left')
    df['exact_dup_of'] = df['exact_dup_of'].fillna('')
    df['dhash'] = df['dhash'].fillna('')
    df['eligible'] = (df.mtcnn_ok == 1) & (df.exact_dup_of == '')
    return df


def build_groups(df, out, celebdf_identity_mode):
    uf = UnionFind()
    for r in df.itertuples(index=False):
        u = 'u:' + r.uid
        uf.find(u)
        if r.video_id:
            uf.union(u, 'v:' + r.video_id)
        if r.dataset == 'DFF':
            if r.rel_key:
                uf.union(u, 'k:' + r.rel_key)          # same source photo across wiki/text2img/inpainting
            if r.identities:
                uf.union(u, 'i:' + r.identities)       # same person
        elif r.dataset == 'CelebDF' and celebdf_identity_mode == 'link':
            for ident in filter(None, r.identities.split(';')):
                uf.union(u, 'i:' + ident)
    pr_path = os.path.join(out, 'near_dup_pairs.csv')
    if os.path.exists(pr_path):
        for r in pd.read_csv(pr_path).itertuples(index=False):
            uf.union('u:' + r.uid_a, 'u:' + r.uid_b)
    return df.uid.map(lambda u: uf.find('u:' + u))


def stage_split(a):
    df = load_eligible(a.out)
    df = df[df.eligible].copy()
    ext = df.dataset.str.startswith('ext_')
    pool = df[~ext].copy()
    pool['group'] = build_groups(pool, a.out, a.celebdf_identity)
    tr, va, te = (float(x) for x in a.ratios.split(','))
    assert abs(tr + va + te - 1) < 1e-6, '--ratios must sum to 1'
    # One split per group, shared by all folds, so a person/video/photo never crosses splits.
    g = pool.groupby('group').agg(forced_test=('celebdf_official_test', lambda s: any(str(x) == '1' for x in s)),
                                  datasets=('dataset', lambda s: '|'.join(sorted(set(s)))))
    rng = random.Random(a.seed)
    split_of = {}
    for gid in sorted(g.index):
        row = g.loc[gid]
        if row.forced_test:
            split_of[gid] = 'test'
        elif 'CelebDF' in row.datasets:
            # CelebDF already has an official test set; remaining videos are divided train/val.
            split_of[gid] = 'val' if rng.random() < va / (tr + va) else 'train'
        else:
            x = rng.random()
            split_of[gid] = 'train' if x < tr else ('val' if x < tr + va else 'test')
    pool['gsplit'] = pool.group.map(split_of)
    sizes = pool.group.value_counts()
    log(f'groups: {len(sizes)}; largest sizes {sizes.head(5).tolist()}; '
        f'singletons {int((sizes == 1).sum())}')
    biggest = sizes.iloc[0] / len(pool)
    if biggest > a.max_group_share:
        sys.exit(f'largest group holds {biggest:.1%} of all images (> {a.max_group_share:.0%}). Linking has '
                 f'chained unrelated images together; inspect near_dup_pairs.csv or lower --near-dup-mad.')
    for s_ in ('train', 'val', 'test'):
        log(f'  group split {s_}: {(pool.gsplit == s_).mean():.1%} of images')
    out_root = os.path.join(a.out, a.split_name)
    os.makedirs(out_root, exist_ok=True)
    matched = a.dff_real == 'matched'
    # Test-only sources. 'original': DFF reals as downloaded train the model; the matched copies are a test-only
    # control. 'matched': the matched copies (stored like the fakes) train the model; the originals become test-only.
    test_only_src = {'dff_insight', 'dff_wiki' if matched else 'dff_wiki_matched'}
    log(f'DFF training reals: {a.dff_real}; test-only sources: {sorted(test_only_src)}; writing {out_root}')
    long_rows = []
    cols = ['crop_path', 'label', 'domain', 'source', 'dataset', 'generator', 'family', 'test_set',
            'uid', 'group', 'video_id', 'identities', 'rel_key']
    for fold, spec in FOLDS.items():
        f = pool.copy()
        role = []
        for r in f.itertuples(index=False):
            gen = r.generator
            if gen in spec['heldout_gens'] or r.source in test_only_src:
                role.append('test' if r.gsplit == 'test' else 'excluded')
            elif r.family == 'real' or gen in spec['train_gens'] or r.dataset in ('HardFake', 'CelebDF'):
                role.append(r.gsplit)
            else:
                role.append('excluded')
        f['role'] = role
        if matched:
            # A matched copy is used only if its original photo is usable here: not an exact duplicate (those were
            # removed before the copies existed) and face-cropped successfully. Otherwise two identical copies could
            # sit on both sides of the split. Rows are excluded after the split, so the groups stay as in E1.
            usable_wiki = set(pool.rel_key[pool.source == 'dff_wiki'])
            orphan = (f.source == 'dff_wiki_matched') & ~f.rel_key.isin(usable_wiki)
            f.loc[orphan, 'role'] = 'excluded'
            if fold == 'A':
                log(f'matched copies excluded (original duplicate or not cropped): {int(orphan.sum())}')
            # the matched reals now carry the DFF (diffusion) domain label; the originals are test-only
            f.loc[f.source == 'dff_wiki_matched', 'domain'] = 1
            f.loc[f.source == 'dff_wiki', 'domain'] = -1
        f['test_set'] = np.where(f.role == 'test', f.dataset + ':' + f.generator, '')
        # CelebDF videos outside the official list can land in test because they share frames (same
        # target clip) with an official test video. Label them so results on the official 518 videos
        # stay comparable with published numbers.
        linked = (f.dataset == 'CelebDF') & (f.role == 'test') & (f.celebdf_official_test.astype(str) != '1')
        f.loc[linked, 'test_set'] = f.loc[linked, 'test_set'] + ':linked'
        # Domain labels only matter for training rows; test-only generators keep d=-1.
        x = df[ext].copy()
        x['group'], x['gsplit'], x['role'] = 'ext:' + x.uid, 'test', 'test'
        x['test_set'] = x.dataset + ':' + x.generator
        f = pd.concat([f, x], ignore_index=True)
        d = os.path.join(out_root, f'fold_{fold}')
        os.makedirs(d, exist_ok=True)
        for s in ('train', 'val', 'test'):
            f.loc[f.role == s, cols].to_csv(os.path.join(d, f'{s}.csv'), index=False)
        log(f'fold {fold}: train={int((f.role == "train").sum())} val={int((f.role == "val").sum())} '
            f'test={int((f.role == "test").sum())} excluded={int((f.role == "excluded").sum())}')
        f['fold'] = fold
        long_rows.append(f[['fold', 'uid', 'role', 'test_set', 'group']])
    pd.concat(long_rows).to_csv(os.path.join(out_root, 'split_manifest_long.csv'), index=False)
    with open(os.path.join(out_root, 'protocol.json'), 'w') as fh:
        json.dump(dict(folds=FOLDS, test_only_sources=sorted(test_only_src), dff_real=a.dff_real,
                       ratios=a.ratios, seed=a.seed, celebdf_identity=a.celebdf_identity), fh, indent=2)


# ----------------------------------------------------------------------------
# Stage 5: audit
# ----------------------------------------------------------------------------
def md_table(df):
    df = df.copy()
    cols = [str(c) for c in df.columns]
    lines = ['| ' + ' | '.join(cols) + ' |', '|' + '---|' * len(cols)]
    for r in df.itertuples(index=False):
        lines.append('| ' + ' | '.join('' if (isinstance(v, float) and np.isnan(v)) else str(v) for v in r) + ' |')
    return '\n'.join(lines)


def overlap_report(fold_df, celebdf_identity_mode, near_pairs=None):
    """Count keys shared between splits. Returns (rows, hard_failures)."""
    rows, failures = [], 0
    keys = {
        'group': lambda s: s.group,
        'video_id': lambda s: s.video_id[s.video_id != ''],
        'identity': lambda s: s.identities[s.identities != ''].str.split(';').explode(),
        'source photo (rel_key, DFF)': lambda s: s.rel_key[(s.dataset == 'DFF') & (s.rel_key != '')],
        'raw file md5': lambda s: s.md5_raw[s.md5_raw != ''],
        'crop md5': lambda s: s.crop_md5[s.crop_md5 != ''],
    }
    by = {s: fold_df[fold_df.role == s] for s in ('train', 'val', 'test')}
    role_of = dict(zip(fold_df.uid, fold_df.role))
    near = near_pairs if near_pairs is not None else pd.DataFrame(columns=['uid_a', 'uid_b'])
    by_noext = {s: v[~v.dataset.str.startswith('ext_')] for s, v in by.items()}
    for name, fn in keys.items():
        for a_, b_ in (('train', 'test'), ('val', 'test'), ('train', 'val')):
            ka, kb = set(fn(by_noext[a_])), set(fn(by_noext[b_]))
            n = len(ka & kb)
            soft = name == 'identity' and celebdf_identity_mode == 'report'
            if n and soft:
                # Only CelebDF identities may overlap in report mode; DFF identities never.
                cdf_only = all(k.startswith('cdf:') for k in ka & kb)
                status = 'WARN (CelebDF identities, report mode)' if cdf_only else 'FAIL'
                failures += 0 if cdf_only else 1
            else:
                status = 'ok' if n == 0 else 'FAIL'
                failures += int(n > 0)
            rows.append(dict(key=name, splits=f'{a_} vs {b_}', shared=n, status=status))
    for a_, b_ in (('train', 'test'), ('val', 'test'), ('train', 'val')):
        n = sum(1 for r in near.itertuples(index=False)
                if {role_of.get(r.uid_a), role_of.get(r.uid_b)} == {a_, b_})
        failures += int(n > 0)
        rows.append(dict(key='near-duplicate pair', splits=f'{a_} vs {b_}', shared=n,
                         status='ok' if n == 0 else 'FAIL'))
    return pd.DataFrame(rows), failures


def stage_audit(a):
    cfg = json.load(open(os.path.join(a.out, 'config.json')))
    cdf_mode = a.celebdf_identity or cfg.get('celebdf_identity', 'report')
    df = load_eligible(a.out)
    split_dir = os.path.join(a.out, a.split_name)
    mf = pd.read_csv(os.path.join(split_dir, 'split_manifest_long.csv'), keep_default_na=False)
    proto = json.load(open(os.path.join(split_dir, 'protocol.json')))
    test_only_src = set(proto.get('test_only_sources', ['dff_insight', 'dff_wiki_matched']))
    real_src = 'dff_wiki_matched' if proto.get('dff_real') == 'matched' else 'dff_wiki'
    rep = os.path.join(a.out, 'reports' if a.split_name == 'splits' else f'reports_{a.split_name}')
    os.makedirs(rep, exist_ok=True)
    md = ['# Phase 1 data audit', '',
          f'Generated {time.strftime("%Y-%m-%d %H:%M")} from `{os.path.abspath(a.out)}`.', '']

    # --- 1. Sample accounting (Comment 2) ---
    unit = df.apply(lambda r: r.video_id if r.video_id else r.raw_path, axis=1)
    acc = df.assign(unit=unit).groupby('source').agg(
        dataset=('dataset', 'first'), generator=('generator', 'first'), label=('label', 'first'),
        raw_files=('unit', 'nunique'), sampled=('uid', 'count'),
        mtcnn_ok=('mtcnn_ok', lambda s: int((s == 1).sum())),
        mtcnn_fail=('mtcnn_ok', lambda s: int((s == 0).sum())),
        exact_dups_removed=('exact_dup_of', lambda s: int((s != '').sum())),
        usable=('eligible', 'sum'))
    for fold in sorted(mf.fold.unique()):
        m = mf[mf.fold == fold].merge(df[['uid', 'source']], on='uid')
        piv = m.pivot_table(index='source', columns='role', values='uid', aggfunc='count', fill_value=0)
        for role in ('train', 'val', 'test', 'excluded'):
            acc[f'{fold}:{role}'] = piv[role] if role in piv else 0
    acc = acc.fillna(0)
    for c in acc.columns:
        if c not in ('dataset', 'generator'):
            acc[c] = acc[c].astype(int)
    acc = acc.reset_index()
    total = acc.select_dtypes('number').sum()
    tot_row = {c: (int(total[c]) if c in total else '') for c in acc.columns}
    tot_row['source'] = 'TOTAL'
    tot_row['label'] = ''
    acc = pd.concat([acc, pd.DataFrame([tot_row])], ignore_index=True)
    acc.to_csv(os.path.join(rep, 'accounting.csv'), index=False)
    md += ['## 1. Sample accounting (R2 Comment 2)', '',
           '`raw_files` counts files (or videos for CelebDF); `sampled` counts images/frames taken from them. '
           'Every later column is computed from the manifest, so the rows add up by construction: '
           '`sampled = mtcnn_ok + mtcnn_fail`, `usable = mtcnn_ok - exact_dups_removed`, and for each fold '
           '`train + val + test + excluded = usable`. `excluded` rows are held-out-generator images whose '
           'person or source photo was assigned to train/val, so they cannot be used as unseen test data.', '',
           md_table(acc), '']
    bad = []
    for rd in acc[acc.source != 'TOTAL'].to_dict('records'):
        if rd['sampled'] != rd['mtcnn_ok'] + rd['mtcnn_fail']:
            bad.append(f'{rd["source"]}: sampled != ok + fail')
        if rd['usable'] != rd['mtcnn_ok'] - rd['exact_dups_removed']:
            bad.append(f'{rd["source"]}: usable mismatch')
        for fold in sorted(mf.fold.unique()):
            if sum(rd[f'{fold}:{x}'] for x in ('train', 'val', 'test', 'excluded')) != rd['usable']:
                bad.append(f'{rd["source"]}: fold {fold} rows != usable')
    md += ['Arithmetic check: ' + ('**all rows add up**' if not bad else '**FAILED** ' + '; '.join(bad)), '']

    # --- 2. Leakage (Comments 3 and 10) ---
    md += ['## 2. Leakage checks (R2 Comments 3 and 10)', '',
           'Shared keys between splits, after removing exact duplicates. `group` is the union of every '
           'link below plus near-duplicates, so a zero there implies zero for each key. Any `FAIL` means '
           'the split is not usable.', '']
    total_fail = 0
    np_path = os.path.join(a.out, 'near_dup_pairs.csv')
    near_pairs = pd.read_csv(np_path) if os.path.exists(np_path) else None
    for fold in sorted(mf.fold.unique()):
        m = mf[mf.fold == fold].drop(columns=['group']).merge(df, on='uid')
        m = m.merge(mf[mf.fold == fold][['uid', 'group']], on='uid')
        if real_src == 'dff_wiki_matched':   # domain labels as written to this split's CSVs
            m.loc[m.source == 'dff_wiki_matched', 'domain'] = 1
            m.loc[m.source == 'dff_wiki', 'domain'] = -1
        rpt, nfail = overlap_report(m, cdf_mode, near_pairs)
        total_fail += nfail
        tr = m[m.role.isin(['train', 'val'])]
        spec = FOLDS[fold]
        leak_gen = (tr.generator.isin(spec['heldout_gens']) | tr.source.isin(test_only_src)).sum()
        total_fail += int(leak_gen > 0)
        md += [f'### Fold {fold}: train on {", ".join(spec["train_gens"])}; held out: '
               f'{", ".join(spec["heldout_gens"])}', '', md_table(rpt), '',
               f'Held-out or test-only generator rows in train/val: **{leak_gen}** (must be 0).', '']
        dom = m[m.role != 'excluded'].groupby(['dataset', 'generator', 'family', 'label', 'domain', 'role']) \
            .size().unstack('role', fill_value=0).reset_index()
        for c in ('train', 'val', 'test'):
            if c not in dom:
                dom[c] = 0
        dom['DAT label used'] = np.where((dom.train > 0) & (dom.domain >= 0), 'yes (d=' + dom.domain.astype(str) + ')',
                                         'no (test only)')
        md += [f'Domain-label table, fold {fold} (R2 Comment 7):', '',
               md_table(dom[['dataset', 'generator', 'family', 'label', 'domain', 'train', 'val', 'test',
                             'DAT label used']]), '']
        if 'CelebDF' in set(m.dataset) and cdf_mode == 'report':
            te_ids = set(m[(m.role == 'test') & (m.dataset == 'CelebDF')].identities.str.split(';').explode()) - {''}
            trv = m[(m.role.isin(['train', 'val'])) & (m.dataset == 'CelebDF')]
            shared = trv.identities.str.split(';').apply(lambda ids: bool(set(ids) & te_ids))
            md += [f'CelebDF identity overlap (report mode): {int(shared.sum())} of {len(trv)} train/val frames '
                   f'({trv[shared].video_id.nunique()} videos) show a person who also appears in the official '
                   f'test list. Videos never cross splits. A fully identity-disjoint CelebDF split is usually not '
                   f'possible: swap pairs connect most of the 59 celebrities into one component, so '
                   f'`--celebdf-identity link` tends to move all of CelebDF into test (the group-size guard '
                   f'stops the run if so). Video-disjoint with the official list is the standard protocol; '
                   f'report this overlap in the paper.', '']
    md += [f'**Overall leakage verdict: {"PASS" if total_fail == 0 else f"FAIL ({total_fail} problems)"}**', '']

    # --- 3. File format / compression shortcuts ---
    fmt = df[df.source != 'dff_wiki_matched'].groupby(['dataset', 'source', 'label']).agg(
        n=('uid', 'count'),
        formats=('orig_format', lambda s: ', '.join(f'{k} {v}' for k, v in Counter(s).most_common(3))),
        median_w=('orig_w', 'median'), median_h=('orig_h', 'median'),
        jpeg_q_mean=('jpeg_q', lambda s: round(pd.to_numeric(s, errors='coerce').mean(), 1))).reset_index()
    fmt.to_csv(os.path.join(rep, 'formats.csv'), index=False)
    warn = []
    for ds, sub in fmt.groupby('dataset'):
        if sub.label.nunique() < 2:
            continue
        f0 = set(sub[sub.label == 0].formats.str.split(' ').str[0])
        f1 = set(sub[sub.label == 1].formats.str.split(' ').str[0])
        if f0 != f1:
            warn.append(f'{ds}: real files are {sorted(f0)}, fake files are {sorted(f1)}')
        q = sub.groupby('label').jpeg_q_mean.mean()
        if q.notna().all() and abs(q.iloc[0] - q.iloc[-1]) > 5:
            warn.append(f'{ds}: mean JPEG quality real {q.iloc[0]:.0f} vs fake {q.iloc[-1]:.0f}')
        s = sub.groupby('label')[['median_w', 'median_h']].mean()
        if len(s) == 2 and not np.allclose(s.iloc[0].values, s.iloc[1].values, rtol=0.1):
            warn.append(f'{ds}: median size real {s.iloc[0].values.tolist()} vs fake {s.iloc[1].values.tolist()}')
    md += ['## 3. File format and size by class', '',
           'A frequency-domain detector can separate classes by compression history alone, so real and fake '
           'images within one dataset should share format, JPEG quality and size. All crops in this rebuild '
           'are saved as lossless PNG, which removes differences introduced by our own pipeline, but '
           'differences already present in the source files survive cropping.', '', md_table(fmt), '',
           ('**Warnings:**\n\n' + '\n'.join(f'- {w}' for w in warn)) if warn else 'No format, quality or size '
           'mismatches between classes.', '']
    md += metadata_baseline(df, mf, rep, real_src)
    with open(os.path.join(rep, 'audit_report.md'), 'w') as fh:
        fh.write('\n'.join(md))
    log(f'audit written to {rep}/audit_report.md  (leakage: {"PASS" if total_fail == 0 else "FAIL"})')
    for w in warn:
        log('  FORMAT WARNING: ' + w)
    if total_fail:
        sys.exit(1)


def matched_quality_estimate():
    """What estimate_jpeg_quality() reads from a file saved like the matched copies (PIL, quality 75)."""
    import io
    buf = io.BytesIO()
    Image.new('RGB', (16, 16)).save(buf, format='JPEG', quality=MATCHED_QUALITY)
    buf.seek(0)
    with Image.open(buf) as im:
        return estimate_jpeg_quality(im)


def metadata_features(d):
    q = pd.to_numeric(d.jpeg_q, errors='coerce')
    if 'source' in d:   # matched copies were recorded as 75.0; use the value measured on a real q75 file
        q = q.where(d.source != 'dff_wiki_matched', matched_quality_estimate())
    fmt = d.orig_format.astype(str).str.upper()
    return np.column_stack([q.fillna(-1), pd.to_numeric(d.orig_w, errors='coerce'),
                            pd.to_numeric(d.orig_h, errors='coerce'),
                            fmt.str.startswith('JPEG'), fmt.str.startswith('PNG')]).astype(float)


def metadata_baseline(df, mf, rep, real_src='dff_wiki'):
    """How well does file metadata alone (JPEG quality, width, height, format) separate DFF reals from
    fakes? A high AUC means the dataset carries a non-face shortcut that the detector could also use."""
    from sklearn.ensemble import HistGradientBoostingClassifier
    from sklearn.metrics import roc_auc_score
    rows = []
    for fold in sorted(mf.fold.unique()):
        spec = FOLDS[fold]
        m = mf[mf.fold == fold].merge(df.drop(columns=['test_set'], errors='ignore'), on='uid')
        other_real = 'dff_wiki' if real_src == 'dff_wiki_matched' else 'dff_wiki_matched'
        m = m[(m.dataset == 'DFF') & (m.source != other_real)]
        tr = m[(m.role == 'train') & ((m.label == 0) | m.generator.isin(spec['train_gens']))]
        if tr.label.nunique() < 2:
            continue
        clf = HistGradientBoostingClassifier(max_iter=200, random_state=0).fit(metadata_features(tr), tr.label)
        te_real = m[(m.role == 'test') & (m.source == real_src)]
        for gen in spec['train_gens'] + spec['heldout_gens'] + ['insightface']:
            te = pd.concat([te_real, m[(m.role == 'test') & (m.generator == gen)]])
            if te.label.nunique() < 2:
                continue
            p = clf.predict_proba(metadata_features(te))[:, 1]
            kind = 'held-out' if gen in spec['heldout_gens'] else ('seen' if gen in spec['train_gens'] else 'test-only')
            rows.append(dict(fold=fold, fakes=f'{gen} ({kind})', n_real=len(te_real), n_fake=int(te.label.sum()),
                             metadata_AUC=round(roc_auc_score(te.label, p), 4)))
    # A one-line rule: "512x512 JPEG at quality ~75 => fake"
    d = df[(df.dataset == 'DFF') & (df.source != 'dff_wiki_matched') & (df.mtcnn_ok == 1)]
    q = pd.to_numeric(d.jpeg_q, errors='coerce')
    rule = (q.between(74, 76)) & (pd.to_numeric(d.orig_w) == 512) & (pd.to_numeric(d.orig_h) == 512)
    rule_txt = (f'Rule "512x512 JPEG with quality 74-76 means fake": flags {rule[d.label == 1].mean():.1%} of DFF '
                f'fakes and {rule[d.label == 0].mean():.1%} of DFF reals.')
    r = pd.DataFrame(rows)
    r.to_csv(os.path.join(rep, 'metadata_baseline.csv'), index=False)
    log('metadata baseline: ' + rule_txt)
    return ['## 4. Metadata-only shortcut baseline (DFF)', '',
            'A gradient-boosted classifier trained on the fold\'s training rows using only file metadata '
            '(JPEG quality, width, height, format), never pixels, and tested on the test rows. An AUC near 1 '
            'means real and fake DFF files can be told apart without looking at a face, so a detector\'s DFF '
            'result needs the matched-real check (`DFF:real_matched` test set) before it can be read as face '
            'forgery detection.', '', md_table(r) if len(r) else '(not enough data)', '', rule_txt, '']


# ----------------------------------------------------------------------------
# Stage 6: audit-original — how leaky was the first-edition split?
# ----------------------------------------------------------------------------
def stage_audit_original(a):
    """Re-create df-1-first-edition cell 9 (stratified 85/15 by (class, domain), random.seed(42)) on the
    same pool of images and measure what the validation set shared with the training set.

    The original lists came from os.walk order on Kaggle, which is not recorded, so the exact
    membership cannot be reproduced; we use sorted paths and repeat over several seeds. Overlap rates
    are properties of an image-level random split, so they barely move between seeds."""
    df = load_merged(a.out)
    hf = df[df.dataset == 'HardFake']                       # originally used uncropped
    dff = df[(df.source.isin(['dff_wiki', 'dff_text2img', 'dff_inpainting'])) & (df.mtcnn_ok == 1)]
    pool = pd.concat([hf, dff], ignore_index=True)
    pool = pool.sort_values(['dataset', 'label', 'raw_path']).reset_index(drop=True)
    near = pd.read_csv(os.path.join(a.out, 'near_dup_pairs.csv')) if os.path.exists(
        os.path.join(a.out, 'near_dup_pairs.csv')) else pd.DataFrame(columns=['uid_a', 'uid_b'])
    res = []
    for seed in [42] + list(range(1, a.repeats)):
        groups = {}
        order = [('HardFake', 0), ('HardFake', 1), ('DFF', 0), ('DFF', 1)]   # notebook insertion order
        for key in order:
            groups[key] = pool[(pool.dataset == key[0]) & (pool.label == key[1])].index.tolist()
        random.seed(seed)
        split = {}
        for key in order:
            items = groups[key]
            random.shuffle(items)
            cut = int(0.85 * len(items))
            for i in items[:cut]:
                split[i] = 'train'
            for i in items[cut:]:
                split[i] = 'val'
        p = pool.assign(split=pool.index.map(split))
        tr, va = p[p.split == 'train'], p[p.split == 'val']
        va_dff = va[va.dataset == 'DFF']
        ids_tr = set(tr.identities) - {''}
        keys_tr = set(tr[tr.dataset == 'DFF'].rel_key)
        va_fake = va_dff[va_dff.label == 1]
        va_real = va_dff[va_dff.label == 0]
        tr_uids, va_uids = set(tr.uid), set(va.uid)
        cross_near = sum(1 for r in near.itertuples(index=False)
                         if (r.uid_a in tr_uids and r.uid_b in va_uids) or (r.uid_b in tr_uids and r.uid_a in va_uids))
        exact = len(set(tr.md5_raw) & set(va.md5_raw) - {''})
        res.append(dict(
            seed=seed, total=len(p), train=len(tr), val=len(va),
            val_dff_identity_in_train=round(100 * va_dff.identities.isin(ids_tr).mean(), 2),
            val_fake_same_photo_in_train=round(100 * va_fake.rel_key.isin(keys_tr).mean(), 2),
            val_real_same_photo_in_train=round(100 * va_real.rel_key.isin(keys_tr).mean(), 2),
            exact_dup_pairs_across=exact, near_dup_pairs_across=cross_near))
    r = pd.DataFrame(res)
    rep = os.path.join(a.out, 'reports')
    os.makedirs(rep, exist_ok=True)
    r.to_csv(os.path.join(rep, 'audit_original.csv'), index=False)
    md = ['# Leakage in the first-edition split', '',
          'Re-creates the stratified 85/15 split from df-1-first-edition (cell 9: groups by (class, domain), '
          '`random.seed(42)`, image-level shuffle) on the same images: HardFake uncropped, DFF wiki/text2img/'
          'inpainting with a successful MTCNN crop. The first edition logged 87,530 images (74,399 train / '
          '13,131 val).', '',
          'Columns, as % of DFF validation images: `val_dff_identity_in_train` = the same person appears in '
          'training; `val_fake_same_photo_in_train` = a validation fake was generated from a real photo whose '
          'own image or other fake version is in training; `val_real_same_photo_in_train` = a validation real '
          'has a fake made from it in training.', '', md_table(r), '']
    with open(os.path.join(rep, 'audit_original.md'), 'w') as fh:
        fh.write('\n'.join(md))
    log('audit-original:\n' + r.to_string(index=False))


# ----------------------------------------------------------------------------
# Phase 2 helper: dataset that reads the split CSVs
# ----------------------------------------------------------------------------
class ManifestDataset:
    """torch Dataset over a split CSV. Returns (image, label, domain, index).

    Usage in the training notebook:
        from cdg_phase1 import ManifestDataset
        ds = ManifestDataset('/kaggle/working/phase1/splits/fold_A/train.csv',
                             root='/kaggle/working/phase1', transform=train_transform)
    """

    def __init__(self, csv_path, root, transform=None):
        self.df = pd.read_csv(csv_path, keep_default_na=False)
        self.root, self.transform = root, transform
        self.paths = [os.path.join(root, p) for p in self.df.crop_path]
        self.labels = self.df.label.astype(int).tolist()
        self.domains = self.df.domain.astype(int).tolist()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[i], self.domains[i], i


# ----------------------------------------------------------------------------
def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = ap.add_subparsers(dest='stage', required=True)

    def common(p):
        p.add_argument('--out', required=True, help='working directory for this rebuild')
        p.add_argument('--seed', type=int, default=2026)
        return p

    p = common(sub.add_parser('scan'))
    p.add_argument('--hardfake-root', help='folder with real/ and fake/')
    p.add_argument('--dff-root', help='folder with wiki/, text2img/, inpainting/ (and optionally insight/)')
    p.add_argument('--dff-matched-real', action='store_true',
                   help='also crop wiki reals after 512x512 + JPEG q75 (test-only shortcut diagnostic)')
    p.add_argument('--dff-insight-sample', type=int, default=0,
                   help='keep a random N of the InsightFace images (test-only); 0 = all')
    p.add_argument('--celebdf-root', help='folder with Celeb-real/, Celeb-synthesis/, YouTube-real/')
    p.add_argument('--celebdf-test-list', help='List_of_testing_videos.txt (default: inside --celebdf-root)')
    p.add_argument('--celebdf-frames-real', type=int, default=10)
    p.add_argument('--celebdf-frames-fake', type=int, default=2,
                   help='fewer frames per fake video, more fake videos: roughly balances classes')
    p.add_argument('--celebdf-max-fake-train-videos', type=int, default=0, help='0 = all')
    p.add_argument('--external', action='append', metavar='NAME=PATH',
                   help='test-only set with PATH/real and/or PATH/fake (repeatable)')
    p.set_defaults(func=stage_scan)

    p = common(sub.add_parser('add-matched'))
    p.set_defaults(func=stage_add_matched)

    p = common(sub.add_parser('crop'))
    p.add_argument('--device', default=None)
    p.add_argument('--limit', type=int, default=0, help='crop at most N rows this call (for testing)')
    p.set_defaults(func=stage_crop)

    p = common(sub.add_parser('dedup'))
    p.add_argument('--near-dup-hamming', type=int, default=6, help='candidate threshold on 64-bit dHash')
    p.add_argument('--near-dup-mad', type=float, default=6.0,
                   help='confirmation threshold: mean abs pixel difference (0-255) on 32x32 grayscale')
    p.set_defaults(func=stage_dedup)

    p = common(sub.add_parser('split'))
    p.add_argument('--dff-real', choices=['original', 'matched'], default='original',
                   help='which DFF reals train the model; matched = stored like the fakes (compression control)')
    p.add_argument('--split-name', default='splits', help='output folder for this split')
    p.add_argument('--ratios', default='0.7,0.1,0.2', help='train,val,test share of groups')
    p.add_argument('--celebdf-identity', choices=['report', 'link'], default='report',
                   help='report (default): video-disjoint, identity overlap reported; link: also '
                        'identity-disjoint (usually collapses CelebDF into one group)')
    p.add_argument('--max-group-share', type=float, default=0.05,
                   help='abort if one linked group exceeds this share of all images')
    p.set_defaults(func=stage_split)

    p = common(sub.add_parser('audit'))
    p.add_argument('--split-name', default='splits')
    p.add_argument('--celebdf-identity', choices=['report', 'link'], default=None)
    p.set_defaults(func=stage_audit)

    p = common(sub.add_parser('audit-original'))
    p.add_argument('--repeats', type=int, default=5, help='seed 42 plus this many-1 other seeds')
    p.set_defaults(func=stage_audit_original)

    a = ap.parse_args(argv)
    if a.stage == 'split':
        # remember the CelebDF mode for the audit stage
        cfg_path = os.path.join(a.out, 'config.json')
        cfg = json.load(open(cfg_path)) if os.path.exists(cfg_path) else {}
        cfg['celebdf_identity'] = a.celebdf_identity
        json.dump(cfg, open(cfg_path, 'w'), indent=2)
    a.func(a)


if __name__ == '__main__':
    main()

In [ ]:
%%writefile cdg_train.py
#!/usr/bin/env python3
"""
CDG-DFD Phase 2 trainer: E1 (held-out generator test), E2 ladder and E3 seeds.

Trains on a Phase 1 fold (train.csv), selects the checkpoint on val.csv only, and evaluates
test.csv exactly once at the end. Every test prediction is saved, so all metrics, confusion
matrices and Figure 4(b) can be regenerated without re-running the model.

Architecture is the paper's (EfficientNet-B4 + SE-style SAM + DAT with GRL), unchanged from
df-1-first-edition except where a flag says otherwise:
  --init imagenet          (default) ImageNet weights, no CelebDF warm start
  --input dct|rgb          DCT (signed log-magnitude, as in the first edition) or raw RGB
  --spectral band|none     revised SAM (attention over DCT frequency bands) on or off
  --channel-attn se        add the first edition's channel attention (squeeze-and-excitation)
  --no-dat                 drop domain-adversarial training
  --lambda-dom, --warmup   DAT weight and GRL warm-up (E8)

Example (Kaggle):
  python cdg_train.py --data /kaggle/input/<phase1-output>/phase1 --fold A --seed 1 --out runs/A_s1
"""
from __future__ import annotations

import argparse
import json
import os
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler


# ----------------------------------------------------------------------------
# Data
# ----------------------------------------------------------------------------
class SplitDataset(Dataset):
    """Holds only numpy arrays (no pandas/Python lists), so forked DataLoader workers do not slowly
    copy the parent's memory. Metadata for reporting is kept separately in .meta (main process only)."""

    def __init__(self, csv_path, root, transform):
        meta = pd.read_csv(csv_path, keep_default_na=False, low_memory=False)
        self.set_rows(meta, root)
        self.transform = transform

    def set_rows(self, meta, root):
        self.meta = meta.reset_index(drop=True)
        self.paths = np.array([os.path.join(root, p) for p in self.meta.crop_path])   # fixed-width str array
        self.labels = self.meta.label.astype(int).to_numpy()
        self.domains = self.meta.domain.astype(int).to_numpy()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(str(self.paths[i])).convert('RGB')
        return self.transform(img), int(self.labels[i]), int(self.domains[i]), i

    def __getstate__(self):           # workers never need the metadata frame
        st = self.__dict__.copy()
        st['meta'] = None
        return st


def make_transforms():
    """CPU side only decodes and converts; the random augmentation runs on the GPU (gpu_augment)."""
    from torchvision import transforms
    base = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])
    return base, base


def _gray(x):
    return (0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3])


def gpu_augment(x, gen=None):
    """First-edition augmentation, batched on the GPU: horizontal flip (p=0.5), rotation in [-10, 10]
    degrees (bilinear, black fill), then brightness/contrast in [0.7, 1.3] and saturation in [0.8, 1.2].
    x: (B, 3, H, W) in [0, 1]."""
    B, dev = x.shape[0], x.device
    u = lambda *s: torch.rand(*s, device=dev, generator=gen)
    flip = u(B) < 0.5
    x = torch.where(flip[:, None, None, None], x.flip(-1), x)
    ang = (u(B) * 2 - 1) * (10 * np.pi / 180)
    cos, sin = torch.cos(ang), torch.sin(ang)
    theta = torch.stack([torch.stack([cos, -sin, torch.zeros_like(cos)], 1),
                         torch.stack([sin, cos, torch.zeros_like(cos)], 1)], 1)
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    x = F.grid_sample(x, grid, mode='bilinear', padding_mode='zeros', align_corners=False)
    b = 0.7 + 0.6 * u(B, 1, 1, 1)
    x = (x * b).clamp(0, 1)
    c = 0.7 + 0.6 * u(B, 1, 1, 1)
    m = _gray(x).mean(dim=(1, 2, 3), keepdim=True)
    x = ((x - m) * c + m).clamp(0, 1)
    s_ = 0.8 + 0.4 * u(B, 1, 1, 1)
    g = _gray(x)
    return ((x - g) * s_ + g).clamp(0, 1)


# ----------------------------------------------------------------------------
# Model (as in df-1-first-edition)
# ----------------------------------------------------------------------------
class BatchDCT(nn.Module):
    """Orthonormal 2-D DCT-II per channel, then sign(D)*log(1+|D|). Always computed in float32."""

    def __init__(self, size=224):
        super().__init__()
        n = torch.arange(size, dtype=torch.float64)
        M = torch.cos(np.pi / size * (n.unsqueeze(0) + 0.5) * n.unsqueeze(1))
        alpha = torch.ones(size, dtype=torch.float64)
        alpha[0] = 1 / np.sqrt(2)
        self.register_buffer('M', (M * alpha.unsqueeze(1) * np.sqrt(2.0 / size)).float())

    def forward(self, x):
        with torch.autocast(device_type=x.device.type, enabled=False):
            D = self.M @ x.float() @ self.M.T
            return torch.sign(D) * torch.log1p(D.abs())


class SpectralAttention(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.conv1 = nn.Conv2d(c, c // 4, 1)
        self.conv2 = nn.Conv2d(c // 4, c, 1)
        self.bn = nn.BatchNorm2d(c // 4)

    def forward(self, x):
        a = torch.sigmoid(self.conv2(F.relu(self.bn(self.conv1(F.adaptive_avg_pool2d(x, 1))))))
        return x * a


class GRL(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, g):
        return g.neg() * ctx.alpha, None


class DomainClassifier(nn.Module):
    def __init__(self, d=1792):
        super().__init__()
        self.classifier = nn.Sequential(nn.Linear(d, 256), nn.ReLU(), nn.Dropout(0.5), nn.Linear(256, 2))

    def forward(self, x, alpha):
        return self.classifier(GRL.apply(x, alpha))


def radial_bands(size=224, bands=16):
    """Band index of every DCT coefficient: normalised radial frequency sqrt(u^2+v^2), split into equal-width
    rings from 0 (DC, lowest frequency) to bands-1 (highest frequency)."""
    k = torch.arange(size, dtype=torch.float64)
    r = torch.sqrt(k[:, None] ** 2 + k[None, :] ** 2) / (np.sqrt(2) * (size - 1))
    return torch.clamp((r * bands).long(), max=bands - 1)


class SpectralBandAttention(nn.Module):
    """Spectral Attention Module (revised): attention over DCT frequency bands, applied to the DCT map
    before the backbone.

    For each image and colour channel it measures the mean magnitude in each of K radial frequency bands,
    passes these C*K numbers through a small MLP, and rescales every coefficient of a band by
    w = 2*sigmoid(z), w in (0, 2). The last layer starts at zero, so w = 1 everywhere at initialisation
    (the module begins as the identity). The weights are returned, so the bands the model relies on can be
    reported per image, per class and per generator."""

    def __init__(self, size=224, bands=16, channels=3, hidden=32):
        super().__init__()
        idx = radial_bands(size, bands)
        onehot = F.one_hot(idx, bands).permute(2, 0, 1).float()          # K, H, W
        self.register_buffer('band_idx', idx, persistent=False)
        self.register_buffer('onehot', onehot, persistent=False)
        self.register_buffer('area', onehot.sum((1, 2)), persistent=False)
        self.bands, self.channels = bands, channels
        self.fc1 = nn.Linear(channels * bands, hidden)
        self.fc2 = nn.Linear(hidden, channels * bands)
        nn.init.zeros_(self.fc2.weight)
        nn.init.zeros_(self.fc2.bias)

    def forward(self, x):
        B, C = x.shape[:2]
        with torch.autocast(device_type=x.device.type, enabled=False):   # small and cheap: keep it in fp32
            xf = x.float()
            energy = torch.einsum('bchw,khw->bck', xf.abs(), self.onehot) / self.area     # B, C, K
            z = self.fc2(F.relu(self.fc1(energy.flatten(1))))
            w = 2 * torch.sigmoid(z).view(B, C, self.bands)
            return xf * w[:, :, self.band_idx], w


class Detector(nn.Module):
    """spectral='band': revised SAM on the DCT map (default). channel_attn='se': the first edition's
    squeeze-and-excitation block on backbone channels (kept for ablation and for loading old checkpoints)."""

    def __init__(self, pretrained=True, spectral='band', channel_attn='off', input_mode='dct', bands=16):
        super().__init__()
        import timm
        self.input_mode = input_mode
        self.dct = BatchDCT(224)
        self.band_attention = SpectralBandAttention(224, bands) if spectral == 'band' else None
        self.backbone = timm.create_model('efficientnet_b4', pretrained=pretrained, num_classes=0, global_pool='')
        # attribute name kept as in the first edition, so old checkpoints still load
        self.spectral_attention = SpectralAttention(1792) if channel_attn == 'se' else nn.Identity()
        self.dropout = nn.Dropout(0.5)
        self.classifier = nn.Sequential(nn.Linear(1792, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, 2))
        self.domain_classifier = DomainClassifier(1792)

    def forward(self, x, alpha=1.0):
        if self.input_mode == 'dct':
            x = self.dct(x)
        if self.band_attention is not None:
            x, w = self.band_attention(x)
            w = w.flatten(1)
        else:
            w = x.new_zeros(x.shape[0], 0)
        x = x.contiguous(memory_format=torch.channels_last)   # much faster depthwise convs on T4
        f = self.spectral_attention(self.backbone(x))
        f = self.dropout(F.adaptive_avg_pool2d(f, 1).flatten(1))
        return self.classifier(f), self.domain_classifier(f, alpha), w


def param_groups(model, wd):
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if p.requires_grad:
            (no_decay if ('bias' in n or 'bn' in n or p.ndim == 1) else decay).append(p)
    return [{'params': decay, 'weight_decay': wd}, {'params': no_decay, 'weight_decay': 0.0}]


# ----------------------------------------------------------------------------
# Metrics
# ----------------------------------------------------------------------------
def roc_auc(y, s):
    from sklearn.metrics import roc_auc_score
    return float(roc_auc_score(y, s)) if len(set(y)) == 2 else float('nan')


def eer(y, s):
    from sklearn.metrics import roc_curve
    if len(set(y)) < 2:
        return float('nan')
    fpr, tpr, _ = roc_curve(y, s)
    fnr = 1 - tpr
    i = int(np.nanargmin(np.abs(fnr - fpr)))
    return float((fpr[i] + fnr[i]) / 2)


def balanced_acc(y, s, thr):
    y, p = np.asarray(y), (np.asarray(s) >= thr).astype(int)
    tpr = (p[y == 1] == 1).mean() if (y == 1).any() else np.nan
    tnr = (p[y == 0] == 0).mean() if (y == 0).any() else np.nan
    return float(np.nanmean([tpr, tnr])), float(tpr), float(tnr)


def bootstrap_auc_ci(y, s, n=1000, seed=0):
    rng = np.random.default_rng(seed)
    y, s = np.asarray(y), np.asarray(s)
    pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
    if not len(pos) or not len(neg):
        return [float('nan')] * 2
    vals = []
    for _ in range(n):   # stratified resampling keeps both classes present
        idx = np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])
        vals.append(roc_auc(y[idx], s[idx]))
    return [float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))]


def evaluation_pairs(test_sets):
    """Which reals are the negatives for which fakes. Returns (name, real sets, fake sets)."""
    sets = set(test_sets)
    pairs = []
    by_ds = {}
    for t in sets:
        ds = t.split(':')[0]
        by_ds.setdefault(ds, []).append(t)
    for ds, ts in sorted(by_ds.items()):
        reals = [t for t in ts if t.split(':')[1] == 'real' and not t.endswith(':linked')]
        reals_all = [t for t in ts if t.split(':')[1] == 'real']
        fakes = sorted(t for t in ts if t.split(':')[1] not in ('real', 'real_matched'))
        for f in fakes:
            if f.endswith(':linked'):
                continue
            pairs.append((f'{f} vs {ds}:real', reals, [f]))
            if ds == 'CelebDF' and any(t.endswith(':linked') for t in ts):
                pairs.append((f'{f} (+linked) vs {ds}:real (+linked)', reals_all,
                              [f] + ([f + ':linked'] if (f + ':linked') in sets else [])))
            if ds == 'DFF' and 'DFF:real_matched' in sets:
                pairs.append((f'{f} vs DFF:real_matched', ['DFF:real_matched'], [f]))
    for t in sorted(sets):
        if t.startswith('ext_') and t.split(':')[1] != 'real' and 'DFF:real' in sets:
            pairs.append((f'{t} vs DFF:real', ['DFF:real'], [t]))
    return pairs


def compute_metrics(pred, thr):
    out = []
    for name, reals, fakes in evaluation_pairs(pred.test_set):
        sub = pred[pred.test_set.isin(reals + fakes)]
        y, s = sub.label.to_numpy(), sub.prob.to_numpy()
        ba, tpr, tnr = balanced_acc(y, s, thr)
        lo, hi = bootstrap_auc_ci(y, s)
        out.append(dict(comparison=name, n_real=int((y == 0).sum()), n_fake=int((y == 1).sum()),
                        AUC=round(roc_auc(y, s), 4), AUC_CI95=f'[{lo:.4f}, {hi:.4f}]', EER=round(eer(y, s), 4),
                        balanced_acc=round(ba, 4), TPR=round(tpr, 4), TNR=round(tnr, 4)))
    # Shortcut check: share of each real set flagged as fake at the chosen threshold.
    for t in ('DFF:real', 'DFF:real_matched'):
        sub = pred[pred.test_set == t]
        if len(sub):
            out.append(dict(comparison=f'{t} flagged as fake', n_real=len(sub), n_fake=0,
                            FPR=round(float((sub.prob >= thr).mean()), 4)))
    return pd.DataFrame(out)


# ----------------------------------------------------------------------------
# Training
# ----------------------------------------------------------------------------
def ram_gb():
    try:
        import psutil
        return psutil.virtual_memory().used / 1e9
    except Exception:  # noqa: BLE001
        return float('nan')


def resources():
    try:
        import psutil
        vm = psutil.virtual_memory()
        out = f'RAM {vm.used / 1e9:.1f}/{vm.total / 1e9:.1f} GB'
    except Exception:  # noqa: BLE001
        out = 'RAM n/a'
    if torch.cuda.is_available():
        out += ' | GPU ' + ', '.join(f'{torch.cuda.max_memory_allocated(i) / 1e9:.1f}'
                                     for i in range(torch.cuda.device_count())) + ' GB peak'
    return out


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


@torch.no_grad()
def predict(model, loader, device, amp, return_weights=False):
    predict.nonfinite, predict.refp32 = 0, 0
    model.eval()
    probs, idxs, ws = [], [], []
    for x, _, _, i in loader:
        xd = x.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
            logits, _, w = model(xd, alpha=0.0)
        p = F.softmax(logits.float(), 1)[:, 1]
        if amp and not torch.isfinite(p).all():   # rare fp16 overflow: score this batch again in fp32
            predict.refp32 += 1
            logits, _, w = model(xd, alpha=0.0)
            p = F.softmax(logits.float(), 1)[:, 1]
        bad = ~torch.isfinite(p)
        if bad.any():   # still not finite in fp32: report, never hide
            predict.nonfinite += int(bad.sum())
            p = torch.where(bad, torch.full_like(p, 0.5), p)
        probs.append(p.cpu())
        ws.append(w.float().cpu())
        idxs.append(i)
    order = torch.cat(idxs).argsort()
    p = torch.cat(probs)[order].numpy()
    if return_weights:
        return p, torch.cat(ws)[order].numpy()
    return p


def val_threshold(y, s):
    """Threshold maximising balanced accuracy on validation (applied unchanged to test)."""
    best, thr = -1, 0.5
    for t in np.unique(np.quantile(s, np.linspace(0.01, 0.99, 197))):
        b = balanced_acc(y, s, t)[0]
        if b > best:
            best, thr = b, float(t)
    return thr


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument('--data', required=True, help='Phase 1 output folder (contains splits/ and crops/)')
    ap.add_argument('--fold', default='A')
    ap.add_argument('--splits', default='splits',
                    help="split folder inside --data: 'splits' (original DFF reals) or 'splits_matched'")
    ap.add_argument('--out', required=True)
    ap.add_argument('--seed', type=int, default=1)
    ap.add_argument('--epochs', type=int, default=20, help='0 = evaluate --init checkpoint only')
    ap.add_argument('--epoch-size', type=int, default=24000, help='samples drawn per epoch (first edition: 24000)')
    ap.add_argument('--batch', type=int, default=64, help='images per GPU step')
    ap.add_argument('--accum', type=int, default=2,
                    help='steps accumulated per update: 64 x 2 = 128, the paper\'s batch size, on one 16 GB T4')
    ap.add_argument('--lr', type=float, default=1e-4)
    ap.add_argument('--max-lr', type=float, default=3e-4)
    ap.add_argument('--wd', type=float, default=2e-4)
    ap.add_argument('--lambda-dom', type=float, default=0.1)
    ap.add_argument('--warmup', type=int, default=10, help='GRL alpha = min(1, epoch/warmup); 0 = constant 1')
    ap.add_argument('--input', choices=['dct', 'rgb'], default='dct')
    ap.add_argument('--spectral', choices=['band', 'none'], default='band',
                    help='band: revised SAM, attention over DCT frequency bands (default)')
    ap.add_argument('--bands', type=int, default=16, help='number of radial DCT bands for --spectral band')
    ap.add_argument('--channel-attn', choices=['off', 'se'], default='off',
                    help="se: first edition's squeeze-and-excitation on backbone channels (ablation / old checkpoints)")
    ap.add_argument('--no-dat', action='store_true', help='drop the domain loss (lambda=0)')
    ap.add_argument('--init', default='imagenet', help="'imagenet', 'none', or a checkpoint path")
    ap.add_argument('--workers', type=int, default=2, help='loader processes (2 runs share 4 CPUs on Kaggle)')
    ap.add_argument('--no-amp', action='store_true')
    ap.add_argument('--gpus', choices=['all', '1'], default='1',
                    help="1 (default): one GPU per run; 'all' = DataParallel (not recommended on Kaggle T4s)")
    ap.add_argument('--pin-memory', action='store_true', help='pinned host memory for the loader (off by default)')
    ap.add_argument('--clip', type=float, default=5.0, help='gradient-norm clipping (0 = off)')
    ap.add_argument('--benchmark', type=int, default=0,
                    help='time this many training batches, write benchmark.json and exit')
    ap.add_argument('--limit-eval', type=int, default=0, help='testing only: cap val/test rows')
    a = ap.parse_args(argv)

    os.makedirs(a.out, exist_ok=True)
    seed_all(a.seed)
    torch.backends.cudnn.benchmark = True
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    amp = device.type == 'cuda' and not a.no_amp
    split_dir = os.path.join(a.data, a.splits, f'fold_{a.fold}')
    tr_tf, te_tf = make_transforms()
    tr = SplitDataset(os.path.join(split_dir, 'train.csv'), a.data, tr_tf)
    va = SplitDataset(os.path.join(split_dir, 'val.csv'), a.data, te_tf)
    te = SplitDataset(os.path.join(split_dir, 'test.csv'), a.data, te_tf)
    if a.limit_eval:
        for d in (va, te):
            # a few rows of every (label, test set) group, so the smoke test exercises every metric
            keep = d.meta.groupby(['label', 'test_set'], group_keys=False).apply(
                lambda g: g.head(max(4, a.limit_eval // 20))).index
            d.set_rows(d.meta.loc[keep], a.data)
    print(f'fold {a.fold}: train={len(tr)} val={len(va)} test={len(te)} | device={device} amp={amp}', flush=True)

    # Same balancing as the first edition: inverse frequency over (label, domain) groups.
    keys = list(zip(tr.labels, tr.domains))
    cnt = pd.Series(keys).value_counts().to_dict()
    weights = torch.tensor([1.0 / cnt[k] for k in keys], dtype=torch.double)
    g = torch.Generator().manual_seed(a.seed)
    sampler = WeightedRandomSampler(weights, num_samples=a.epoch_size, replacement=True, generator=g)
    pin = device.type == 'cuda' and a.pin_memory
    tl = DataLoader(tr, batch_size=a.batch, sampler=sampler, drop_last=True, num_workers=a.workers,
                    pin_memory=pin, persistent_workers=a.workers > 0, prefetch_factor=4 if a.workers else None)
    eval_kw = dict(batch_size=a.batch, shuffle=False, num_workers=a.workers, pin_memory=pin, persistent_workers=False)
    vl = DataLoader(va, **eval_kw)

    pretrained = a.init == 'imagenet'
    model = Detector(pretrained=pretrained, spectral=a.spectral, channel_attn=a.channel_attn,
                     input_mode=a.input, bands=a.bands)
    n_band = sum(p.numel() for p in model.band_attention.parameters()) if model.band_attention is not None else 0
    print(f'parameters: total {sum(p.numel() for p in model.parameters()):,} | band attention {n_band:,}', flush=True)
    if a.init not in ('imagenet', 'none'):
        sd = torch.load(a.init, map_location='cpu', weights_only=True)
        miss, unexp = model.load_state_dict(sd, strict=False)
        print(f'init from {a.init}: missing={len(miss)} unexpected={len(unexp)}')
    model = model.to(device)
    model = model.to(memory_format=torch.channels_last)
    net = nn.DataParallel(model) if (torch.cuda.device_count() > 1 and a.gpus == 'all') else model
    print(f'GPUs used: {torch.cuda.device_count() if net is not model else 1}', flush=True)
    opt = torch.optim.AdamW(param_groups(model, a.wd), lr=a.lr)
    steps_per_epoch = len(tl) // a.accum
    sched = (torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=a.max_lr, total_steps=a.epochs * steps_per_epoch)
             if a.epochs > 0 else None)
    scaler = torch.amp.GradScaler('cuda', enabled=amp)
    lam = 0.0 if a.no_dat else a.lambda_dom
    cfg = dict(vars(a), effective_batch=a.batch * a.accum, n_train=len(tr), n_val=len(va), n_test=len(te), group_counts={str(k): v for k, v in cnt.items()},
               torch=torch.__version__, gpu=torch.cuda.get_device_name(0) if device.type == 'cuda' else 'cpu')
    json.dump(cfg, open(os.path.join(a.out, 'config.json'), 'w'), indent=2)

    history, best_auc, t0, start_epoch = [], -1.0, time.time(), 0
    last_path = os.path.join(a.out, 'last.pth')
    aug_gen = torch.Generator(device=device).manual_seed(a.seed + 1000)
    if a.epochs > 0 and os.path.exists(last_path):   # resume after a crash or a killed process
        ck = torch.load(last_path, map_location='cpu', weights_only=False)   # RNG states must stay on the CPU
        model.load_state_dict(ck['model'])
        opt.load_state_dict(ck['opt'])        # moves optimizer state to the parameters' device
        sched.load_state_dict(ck['sched'])
        scaler.load_state_dict(ck['scaler'])
        g.set_state(ck['sampler_gen'].cpu())
        aug_gen.set_state(ck['aug_gen'].cpu())
        history, best_auc, start_epoch = ck['history'], ck['best_auc'], ck['epoch']
        del ck
        print(f'resumed from {last_path} after epoch {start_epoch} (best val AUC {best_auc:.4f})', flush=True)
    if a.epochs == 0:   # evaluation only: score an existing checkpoint (--init) on this fold's val and test
        torch.save(model.state_dict(), os.path.join(a.out, 'best.pth'))
        pv = predict(net, vl, device, amp)
        np.save(os.path.join(a.out, 'val_probs_best.npy'), pv)
        best_auc = roc_auc(va.labels, pv)
        history.append(dict(epoch=0, val_auc=best_auc))
    for epoch in range(start_epoch, a.epochs):
        net.train()
        alpha = 1.0 if a.warmup == 0 else min(1.0, epoch / a.warmup)
        tot, correct, n, dom_correct = 0.0, 0, 0, 0
        te0, wait, nb, bad_steps, group_bad = time.time(), 0.0, 0, 0, False
        opt.zero_grad(set_to_none=True)
        tw = time.time()
        for x, y, d, _ in tl:
            wait += time.time() - tw
            nb += 1
            x, y, d = x.to(device, non_blocking=True), y.to(device), d.to(device)
            x = gpu_augment(x, aug_gen)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
                cls, dom, _ = net(x, alpha=alpha)
                loss = F.cross_entropy(cls, y)
                if lam > 0:
                    loss = loss + lam * F.cross_entropy(dom, d)
            if torch.isfinite(loss):
                scaler.scale(loss / a.accum).backward()
                tot += loss.item() * len(y)
                n += len(y)
                correct += (cls.argmax(1) == y).sum().item()
                dom_correct += (dom.argmax(1) == d).sum().item()
            else:
                bad_steps += 1
                group_bad = True
                if bad_steps > 50:
                    raise RuntimeError('more than 50 training steps with a non-finite loss: training diverged')
            if nb % a.accum == 0:   # one optimizer update per a.accum batches (effective batch a.batch * a.accum)
                if group_bad:
                    opt.zero_grad(set_to_none=True)
                else:
                    if a.clip > 0:
                        scaler.unscale_(opt)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), a.clip)
                    scaler.step(opt)
                    scaler.update()
                    opt.zero_grad(set_to_none=True)
                group_bad = False
                sched.step()
            if nb == min(10, max(a.benchmark // 3, 1)) or (nb == 10 and not a.benchmark):
                ram0, t_first, nb0 = ram_gb(), time.time(), nb   # after warm-up (optimizer state, cudnn tuning)
            if nb % 20 == 0 and (nb <= 100 or nb % 100 == 0):
                el = time.time() - te0
                print(f'   batch {nb}/{len(tl)}: {nb * a.batch / el:.0f} img/s, waiting for data {100 * wait / el:.0f}% '
                      f'of the time | {resources()}', flush=True)
            if a.benchmark and nb == a.benchmark:
                el = time.time() - t_first
                res = dict(gpus=a.gpus, pin_memory=a.pin_memory, batches=nb, img_per_s=round((nb - nb0) * a.batch / el, 1),
                           ram_growth_mb_per_batch=round(1000 * (ram_gb() - ram0) / max(nb - nb0, 1), 1),
                           data_wait_pct=round(100 * wait / (time.time() - te0), 1), skipped_nonfinite=bad_steps)
                json.dump(res, open(os.path.join(a.out, 'benchmark.json'), 'w'))
                print('BENCHMARK', json.dumps(res), flush=True)
                return
            tw = time.time()
        opt.zero_grad(set_to_none=True)   # drop a leftover partial accumulation at the end of the epoch
        train_min = (time.time() - te0) / 60
        if n == 0:
            raise RuntimeError('every training step this epoch had a non-finite loss')
        if bad_steps:
            print(f'   WARNING: skipped {bad_steps} steps with a non-finite loss this epoch', flush=True)
        pv = predict(net, vl, device, amp)
        if predict.nonfinite:
            print(f'   WARNING: {predict.nonfinite} validation scores were not finite (set to 0.5)', flush=True)
        if predict.refp32:
            print(f'   note: {predict.refp32} validation batches re-scored in fp32 after an fp16 overflow', flush=True)
        yv, dv = va.labels, va.domains
        row = dict(epoch=epoch + 1, alpha=round(alpha, 2), loss=tot / n, train_acc=correct / n,
                   train_domain_acc=dom_correct / n, val_auc=roc_auc(yv, pv),
                   val_bal_acc=balanced_acc(yv, pv, 0.5)[0], minutes=(time.time() - t0) / 60,
                   train_minutes=train_min, data_wait_pct=100 * wait / max(time.time() - te0, 1e-9),
                   skipped_steps=bad_steps, val_nonfinite=predict.nonfinite)
        for dom_id, name in ((0, 'gan'), (1, 'diff')):
            m = dv == dom_id
            row[f'val_auc_{name}'] = roc_auc(yv[m], pv[m]) if m.any() else float('nan')
        history.append(row)
        pd.DataFrame(history).to_csv(os.path.join(a.out, 'history.csv'), index=False)
        flag = ''
        if row['val_auc'] > best_auc:   # selection on validation AUC only; test is never touched here
            best_auc = row['val_auc']
            torch.save(model.state_dict(), os.path.join(a.out, 'best.pth'))
            np.save(os.path.join(a.out, 'val_probs_best.npy'), pv)
            flag = '  * best'
        print(f"ep {row['epoch']:02d} a={row['alpha']:.2f} loss={row['loss']:.4f} train={row['train_acc']:.3f} "
              f"dom={row['train_domain_acc']:.3f} | val AUC={row['val_auc']:.4f} (GAN {row['val_auc_gan']:.4f}, "
              f"diff {row['val_auc_diff']:.4f}) balacc={row['val_bal_acc']:.3f} | {row['minutes']:.1f} min{flag}",
              flush=True)
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                        scaler=scaler.state_dict(), sampler_gen=g.get_state().cpu(),
                        aug_gen=aug_gen.get_state().cpu(),
                        history=history, best_auc=best_auc, epoch=epoch + 1), last_path + '.tmp')
        os.replace(last_path + '.tmp', last_path)   # atomic: a kill mid-save never corrupts it

    # ---- single test evaluation with the selected checkpoint ----
    model.load_state_dict(torch.load(os.path.join(a.out, 'best.pth'), map_location=device, weights_only=True))
    pv = np.load(os.path.join(a.out, 'val_probs_best.npy'))
    thr = val_threshold(va.labels, pv)
    tel = DataLoader(te, **eval_kw)
    pt, wt = predict(net, tel, device, amp, return_weights=True)
    test_nonfinite, test_refp32 = predict.nonfinite, predict.refp32
    pred = te.meta[['uid', 'test_set', 'label', 'source', 'generator']].copy()
    pred['prob'] = pt
    if wt.shape[1]:   # band weights: mean per test set, per colour channel and band (for the SAM figure)
        np.save(os.path.join(a.out, 'test_band_weights.npy'), wt.astype(np.float16))
        K = model.band_attention.bands
        cols = [f'{c}_b{k:02d}' for c in 'RGB' for k in range(K)]
        bw = pd.DataFrame(wt, columns=cols)
        bw['test_set'] = pred.test_set.values
        bw.groupby('test_set').mean().round(4).to_csv(os.path.join(a.out, 'band_weights_by_test_set.csv'))
    pred.to_csv(os.path.join(a.out, 'test_predictions.csv'), index=False)
    met = compute_metrics(pred, thr)
    met.to_csv(os.path.join(a.out, 'test_metrics.csv'), index=False)
    hist = pd.DataFrame(history)
    best_ep = int(hist.loc[hist.val_auc.idxmax(), 'epoch'])
    summary = dict(fold=a.fold, seed=a.seed, splits=a.splits, best_epoch=best_ep, best_val_auc=best_auc,
                   threshold=thr, minutes=(time.time() - t0) / 60, test_nonfinite=test_nonfinite,
                   test_batches_rescored_fp32=test_refp32)
    json.dump(summary, open(os.path.join(a.out, 'summary.json'), 'w'), indent=2)
    print(json.dumps(summary, indent=2))
    if os.path.exists(last_path):
        os.remove(last_path)   # finished: keep only best.pth
    with pd.option_context('display.width', 200, 'display.max_columns', 20):
        print(met.to_string(index=False))


if __name__ == '__main__':
    main()

In [ ]:
%%writefile cdg_external.py
#!/usr/bin/env python3
"""
CDG-DFD external test: score trained checkpoints on DiffusionFace (DiffFace; Chen et al., 2024; Zenodo, CC BY 4.0),
whose generators were never used in training.

Stages (all outputs under --out):
  fetch   download each archive, keep a random sample of N images, delete the archive   -> raw/<set>/...
  crop    MTCNN crop (Phase 1 settings) in two versions:                                  -> crops/<view>/<set>/...
            asis     the files as distributed
            matched  every image (real and fake) first resized to 512x512 and saved as JPEG quality 75,
                     the DFF fakes' storage, so both classes share one file history
  score   every checkpoint (best.pth + config.json) on every crop                        -> scores.csv
  report  AUC of each generator against DiffFace reals, per view, mean and spread over seeds,
          plus a file-format table per set                                               -> report.md, report.csv
"""
from __future__ import annotations

import argparse
import glob
import hashlib
import io
import json
import os
import random
import sys
import tarfile
import time
import urllib.request

import numpy as np
import pandas as pd
from PIL import Image

IMG_EXT = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')
ZENODO = {'part1': '10865300', 'part2': '10865065'}   # record ids
URL_FORMS = ['https://zenodo.org/records/{rec}/files/{f}?download=1',
             'https://zenodo.org/api/records/{rec}/files/{f}/content',
             'https://zenodo.org/record/{rec}/files/{f}?download=1']
# name -> (zenodo part, archive, label, family)
SETS = {
    'Real':           ('part1', 'Real.tar', 0, 'real'),
    'DDPM':           ('part1', 'DDPM.tar', 1, 'unconditional diffusion'),
    'DDIM':           ('part1', 'DDIM.tar', 1, 'unconditional diffusion'),
    'PNDM':           ('part1', 'PNDM.tar', 1, 'unconditional diffusion'),
    'LDM':            ('part1', 'LDM.tar', 1, 'latent diffusion'),
    'ADM':            ('part1', 'ADM.tar', 1, 'unconditional diffusion'),
    'SDv21_img2img':  ('part1', 'SDv21_DS0.5.tar', 1, 'Stable Diffusion 2.1 (image-to-image)'),
    'SDv21_text2img': ('part2', 'stable_diffusion_v_2_1_text2img_p1g7.tar', 1, 'Stable Diffusion 2.1 (text-to-image)'),
}
MATCHED_SIZE, MATCHED_QUALITY = (512, 512), 75


def log(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True)


# ----------------------------------------------------------------------------
def _why(url):
    """First bytes of the server's answer, to explain a refusal."""
    try:
        with urllib.request.urlopen(url, timeout=30) as r:
            return f'HTTP {r.status}, {r.headers.get("Content-Type")}'
    except urllib.error.HTTPError as e:
        body = e.read(300).decode('utf-8', 'replace').replace('\n', ' ')
        return f'HTTP {e.code}: {body[:200]}'
    except Exception as e:  # noqa: BLE001
        return f'{type(e).__name__}: {e}'


def download(rec, fname, dest):
    """Try plain download tools (no browser identity) and several Zenodo URL forms; keep the first real tar."""
    import shutil
    import subprocess
    tmp = dest + '.part'
    tried = []
    for form in URL_FORMS:
        url = form.format(rec=rec, f=fname)
        tools = []
        if shutil.which('curl'):
            tools.append(('curl', ['curl', '-fL', '--retry', '3', '--retry-delay', '5', '-sS', '-o', tmp, url]))
        if shutil.which('wget'):
            tools.append(('wget', ['wget', '-q', '--tries=3', '-O', tmp, url]))
        tools.append(('python', None))
        for tool, cmd in tools:
            if os.path.exists(tmp):
                os.remove(tmp)
            t0 = time.time()
            try:
                if cmd:
                    subprocess.run(cmd, check=True, timeout=3 * 3600)
                else:
                    with urllib.request.urlopen(url, timeout=120) as r, open(tmp, 'wb') as f:
                        shutil.copyfileobj(r, f, 1 << 22)
                if tarfile.is_tarfile(tmp):
                    os.replace(tmp, dest)
                    gb = os.path.getsize(dest) / 1e9
                    log(f'   {fname}: {gb:.2f} GB with {tool} in {(time.time() - t0) / 60:.1f} min')
                    return
                tried.append(f'{tool} {url}: got a non-tar file ({os.path.getsize(tmp)} bytes)')
            except Exception as e:  # noqa: BLE001
                tried.append(f'{tool} {url}: {type(e).__name__}')
    if os.path.exists(tmp):
        os.remove(tmp)
    raise RuntimeError('all download attempts failed:\n   ' + '\n   '.join(tried) +
                       f'\n   server says: {_why(URL_FORMS[0].format(rec=rec, f=fname))}')


class _Meter:
    """File-like wrapper: counts bytes, reports speed every 30 s, stops at a deadline."""

    def __init__(self, f, label, deadline):
        self.f, self.label, self.deadline = f, label, deadline
        self.n, self.t0, self.last = 0, time.time(), time.time()

    def read(self, size=-1):
        if time.time() > self.deadline:
            raise TimeoutError('time limit for this set reached')
        b = self.f.read(size)
        self.n += len(b)
        if time.time() - self.last > 30:
            self.last = time.time()
            log(f'   {self.label}: {self.n / 1e9:.2f} GB read, {self.n / 1e6 / (time.time() - self.t0):.1f} MB/s')
        return b


def stream_sample(rec, fname, out_dir, n, stride, minutes):
    """Read the .tar while it downloads and stop once n images are kept (every stride-th image in archive order),
    so only a fraction of the archive is transferred. Returns (kept, note)."""
    agents = [None, 'curl/8.5.0', 'Wget/1.21.4']   # never a browser identity (those get bot challenges)
    notes = []
    for form in URL_FORMS:
        url = form.format(rec=rec, f=fname)
        for ua in agents:
            req = urllib.request.Request(url, headers={'User-Agent': ua} if ua else {})
            try:
                r = urllib.request.urlopen(req, timeout=60)
            except Exception as e:  # noqa: BLE001
                notes.append(f'{url} [{ua or "python"}]: {getattr(e, "code", type(e).__name__)}')
                continue
            os.makedirs(out_dir, exist_ok=True)
            kept, seen, meter = 0, 0, _Meter(r, fname, time.time() + 60 * minutes)
            try:
                with tarfile.open(fileobj=meter, mode='r|') as tf:
                    for m in tf:
                        if not (m.isfile() and m.name.lower().endswith(IMG_EXT)) or \
                                os.path.basename(m.name).startswith('.'):
                            continue
                        seen += 1
                        if (seen - 1) % stride:
                            continue
                        data = tf.extractfile(m).read()
                        key = hashlib.sha1(m.name.encode()).hexdigest()[:16]
                        with open(os.path.join(out_dir, key + os.path.splitext(m.name)[1].lower()), 'wb') as f:
                            f.write(data)
                        kept += 1
                        if kept >= n:
                            break
                note = 'complete'
            except TimeoutError:
                note = f'stopped at the {minutes}-minute limit'
            except Exception as e:  # noqa: BLE001
                note = f'stream ended early ({type(e).__name__})'
            finally:
                r.close()
            log(f'   {fname}: kept {kept} of the first {seen} images, {meter.n / 1e9:.2f} GB read in '
                f'{(time.time() - meter.t0) / 60:.1f} min ({note})')
            return kept, note
    return 0, 'no URL accepted: ' + '; '.join(notes)


def stage_fetch(a):
    failed = []
    for name in a.sets:
        rng = random.Random(f'{a.seed}:{name}')   # per set, so one failed set does not change the others
        part, archive, label, family = SETS[name]
        out_dir = os.path.join(a.out, 'raw', name)
        if os.path.isdir(out_dir) and len(os.listdir(out_dir)) >= a.min_keep:
            log(f'{name}: already sampled ({len(os.listdir(out_dir))} files)')
            continue
        src = os.path.join(a.local, archive) if a.local else None
        tar_path = src if src and os.path.exists(src) else os.path.join(a.out, archive)
        if not os.path.exists(tar_path):
            # stream from Zenodo and stop early (about n*stride images are transferred, not the whole archive)
            log(f'{name}: streaming {archive} from Zenodo')
            kept, note = stream_sample(ZENODO[part], archive, out_dir, a.n, a.stride, a.minutes)
            if kept < a.min_keep:
                log(f'{name}: FAILED ({kept} images; {note})')
                failed.append(name)
                import shutil
                shutil.rmtree(out_dir, ignore_errors=True)   # a partial set is not used
            continue
        log(f'{name}: indexing {archive}')
        with tarfile.open(tar_path) as tf:
            members = [m for m in tf.getmembers() if m.isfile() and m.name.lower().endswith(IMG_EXT)
                       and not os.path.basename(m.name).startswith('.')]
            members.sort(key=lambda m: m.name)
            pick = rng.sample(members, min(a.n, len(members)))
            os.makedirs(out_dir, exist_ok=True)
            for m in pick:
                data = tf.extractfile(m).read()
                key = hashlib.sha1(m.name.encode()).hexdigest()[:16]
                with open(os.path.join(out_dir, key + os.path.splitext(m.name)[1].lower()), 'wb') as f:
                    f.write(data)
        log(f'{name}: {len(pick)} of {len(members)} images kept')
        if tar_path.startswith(a.out):
            os.remove(tar_path)   # keep the disk free
    have = [n for n in a.sets if os.path.isdir(os.path.join(a.out, 'raw', n)) and os.listdir(os.path.join(a.out, 'raw', n))]
    log(f'sampled sets: {have}; failed: {failed}')
    if 'Real' not in have or len(have) < 2:
        sys.exit('Not enough sets downloaded (need Real and at least one generator). If Zenodo refuses Kaggle, '
                 'download Real.tar and a few generator .tar files in a browser, upload them as a Kaggle dataset, '
                 'attach it, and run again: the notebook passes it with --local.')


# ----------------------------------------------------------------------------
# ArtiFact (Rahman et al., ICIP 2023; Kaggle awsaf49/artifact-dataset): already on Kaggle, attach as an input.
# One folder per source, each with metadata.csv (image_path, target 0/1, category). Every image (real and fake)
# went through the same random crop, downscale and JPEG impairments and is 200x200, so the two classes share
# one file history by construction (no compression shortcut).
AF_REAL = ('ffhq', 'celebahq')                     # real face sources
AF_FAMILY = {
    'ddpm': 'diffusion (not SD)', 'latent_diffusion': 'diffusion (not SD)', 'glide': 'diffusion (not SD)',
    'denoising_diffusion_gan': 'diffusion-GAN hybrid', 'diffusion_gan': 'diffusion-GAN hybrid',
    'vq_diffusion': 'diffusion (not SD)', 'palette': 'diffusion (not SD)',
    'stable_diffusion': 'Stable Diffusion (seen family)',
    'stylegan1': 'GAN', 'stylegan2': 'GAN', 'stylegan3': 'GAN', 'star_gan': 'GAN', 'stargan': 'GAN', 'cips': 'GAN',
    'projected_gan': 'GAN', 'gansformer': 'GAN', 'big_gan': 'GAN', 'biggan': 'GAN', 'pro_gan': 'GAN',
    'progan': 'GAN', 'cycle_gan': 'GAN', 'gau_gan': 'GAN', 'taming_transformer': 'transformer',
    'generative_inpainting': 'inpainting', 'lama': 'inpainting', 'mat': 'inpainting',
    'sfhq': 'mixed (SFHQ)', 'face_synthetics': 'rendered (3D)',
}
# Face subsets, from the folder inventory on Kaggle (category = the generator's training set or prompt group)
_ALL = lambda c: True
AF_FACES = {
    'ffhq': _ALL, 'celebahq': _ALL,                                   # reals
    'stylegan1': _ALL,                                               # '1 million fake faces'
    'stylegan2': lambda c: c == 'ffhq',
    'stylegan3': lambda c: 'ffhq' in c.lower(),
    'cips': lambda c: c == 'cips-ffhq',
    'gansformer': lambda c: c == 'ffhq_images',
    'projectedgan': lambda c: c == 'ffhq',
    'tamingtransformer': lambda c: c == 'tt-ffhq',
    'diffusiongan': lambda c: 'ffhq' in c.lower(),                   # Diffusion-StyleGAN2-FFHQ
    'denoisingdiffusiongan': _ALL,                                   # content not labelled: MTCNN rate shows it
    'lama': lambda c: c == 'celeb',                                  # inpainted CelebA-HQ
    'stargan': _ALL, 'sfhq': _ALL, 'facesynthetics': _ALL,
    'stablediffusion': lambda c: c in ('Male', 'Female'),
}
FACE_RX = r'face|celeb|ffhq|portrait'
FACE_RX2 = r'human|person|people'


def _norm(x):
    return x.lower().replace('_', '').replace('-', '')


def af_family(folder):
    k = _norm(folder)
    if k in [_norm(x) for x in AF_REAL] or k in ('imagenet', 'coco', 'lsun', 'afhq', 'metfaces', 'landscape'):
        return 'real'
    return {_norm(x): v for x, v in AF_FAMILY.items()}.get(k, 'other')


def find_artifact(roots, max_depth=6):
    """Folders holding metadata.csv under the ArtiFact input; glob by fixed depth (a recursive glob would walk
    2.5 million files)."""
    found = []
    for root in roots:
        for d in range(1, max_depth + 1):
            found += glob.glob(os.path.join(root, *(['*'] * d), 'metadata.csv'))
    return sorted(set(found))


def _resolve(folder, rel):
    for c in (os.path.join(folder, rel), os.path.join(os.path.dirname(folder), rel),
              os.path.join(folder, os.path.basename(rel))):
        if os.path.exists(c):
            return c
    parts = rel.replace('\\', '/').split('/')
    for k in range(1, len(parts)):   # path written from a different root: drop leading parts
        for base in (folder, os.path.dirname(folder)):
            c = os.path.join(base, *parts[k:])
            if os.path.exists(c):
                return c
    return None


def stage_artifact(a):
    import shutil
    metas = find_artifact(a.artifact_root)
    log(f'{len(metas)} metadata.csv files found under {a.artifact_root}')
    if not metas:
        sys.exit('ArtiFact not found: attach the Kaggle dataset "awsaf49/artifact-dataset" as an input')
    summary, used = [], set()
    for mpath in metas:
        folder = os.path.dirname(mpath)
        name = os.path.basename(folder)
        if name in used:   # same folder name at two places: add the parent's name
            name = os.path.basename(os.path.dirname(folder)) + '_' + name
        used.add(name)
        fam = af_family(name)
        try:
            m = pd.read_csv(mpath, keep_default_na=False, dtype=str)
        except Exception as e:  # noqa: BLE001
            log(f'{name}: metadata unreadable ({type(e).__name__}), skipped'); continue
        if 'category' not in m or 'image_path' not in m:
            log(f'{name}: no category/image_path column, skipped'); continue
        cats = m['category'].value_counts()
        row = dict(folder=name, family=fam, rows=len(m),
                   categories='; '.join(f'{k} {v}' for k, v in cats.head(8).items()))
        if 'target' in m:   # 0 = real; ArtiFact numbers the fake generator types (1, 2, 6, ...)
            t = m['target'].astype(str).str.strip()
            m = m[(t == '0') if fam == 'real' else (t != '0')]
        sel = AF_FACES.get(_norm(name))
        if sel is None:
            summary.append(dict(row, face_rows=0, kept=0, note='not a face source')); continue
        face = m[[bool(sel(str(c))) for c in m['category']]]
        row['face_rows'] = len(face)
        n_want = a.n_real if fam == 'real' else a.n
        if face.empty:
            summary.append(dict(row, kept=0, note='no face images')); continue
        pick = face.sample(n=min(n_want, len(face)), random_state=a.seed)
        set_name = 'Real' if fam == 'real' else 'AF_' + name
        out_dir = os.path.join(a.out, 'raw', set_name)
        os.makedirs(out_dir, exist_ok=True)
        kept = 0
        for rel in pick['image_path']:
            p = _resolve(folder, rel)
            if p is None:
                continue
            key = (name + '_' if fam == 'real' else '') + hashlib.sha1(rel.encode()).hexdigest()[:16]
            dst = os.path.join(out_dir, key + os.path.splitext(p)[1].lower())
            if not os.path.exists(dst):
                shutil.copyfile(p, dst)
            kept += 1
        summary.append(dict(row, kept=kept, note='' if kept else 'image_path did not resolve'))
        log(f'{name} [{fam}]: {len(face):,} face images of {len(m):,}; kept {kept}')
    s = pd.DataFrame(summary)
    s.to_csv(os.path.join(a.out, 'artifact_inventory.csv'), index=False)
    pd.set_option('display.width', 250); pd.set_option('display.max_colwidth', 90)
    print(s.to_string(index=False), flush=True)
    sets = sorted(os.listdir(os.path.join(a.out, 'raw'))) if os.path.isdir(os.path.join(a.out, 'raw')) else []
    json.dump({x: (0 if x == 'Real' else 1) for x in sets}, open(os.path.join(a.out, 'sets.json'), 'w'), indent=1)
    if 'Real' not in sets or len(sets) < 2:
        sys.exit('ArtiFact: no real faces or no fake faces were found; the inventory above shows what is there')


# ----------------------------------------------------------------------------
def estimate_jpeg_quality(im):
    from cdg_phase1 import estimate_jpeg_quality as est   # same estimator as Phase 1
    return est(im)


def to_matched(img):
    buf = io.BytesIO()
    img.resize(MATCHED_SIZE, Image.Resampling.BICUBIC).save(buf, format='JPEG', quality=MATCHED_QUALITY)
    buf.seek(0)
    return Image.open(buf).convert('RGB')


def stage_crop(a):
    import torch
    from facenet_pytorch import MTCNN
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    mtcnn = MTCNN(image_size=224, margin=40, post_process=False, keep_all=False, device=device)
    rows = []
    sets = a.sets or sorted(os.listdir(os.path.join(a.out, 'raw')))
    for name in sets:
        files = sorted(glob.glob(os.path.join(a.out, 'raw', name, '*')))
        if not files:
            log(f'{name}: not downloaded, skipped')
            continue
        n_ok = {'asis': 0, 'matched': 0}
        for p in files:
            uid = os.path.splitext(os.path.basename(p))[0]
            try:
                with Image.open(p) as im:
                    fmt, w, h = im.format, im.size[0], im.size[1]
                    q = estimate_jpeg_quality(im)
                    img = im.convert('RGB')
            except Exception as e:  # noqa: BLE001
                rows.append(dict(set=name, uid=uid, view='asis', ok=0, reason=f'read:{type(e).__name__}'))
                continue
            for view, im_v in (('asis', img), ('matched', to_matched(img))):
                out = os.path.join(a.out, 'crops', view, name, uid + '.png')
                ok = os.path.exists(out)
                if not ok:
                    face = mtcnn(im_v)
                    if face is not None:
                        os.makedirs(os.path.dirname(out), exist_ok=True)
                        Image.fromarray(face.permute(1, 2, 0).clamp(0, 255).byte().cpu().numpy()).save(out)
                        ok = True
                n_ok[view] += int(ok)
                rows.append(dict(set=name, uid=uid, view=view, ok=int(ok), crop=out if ok else '',
                                 orig_format=fmt, orig_w=w, orig_h=h, jpeg_q=q))
        log(f'{name}: {len(files)} images, faces found asis {n_ok["asis"]}, matched {n_ok["matched"]}')
    pd.DataFrame(rows).to_csv(os.path.join(a.out, 'crops.csv'), index=False)


# ----------------------------------------------------------------------------
def _walk_inputs(roots):
    """(best.pth paths, zip paths) under roots, without entering image datasets: a folder whose sub-folders hold
    metadata.csv (ArtiFact) or that has thousands of entries is not searched."""
    pths, zips = [], []
    for root in roots:
        for d, dirs, files in os.walk(root, followlinks=True):
            if 'metadata.csv' in files or len(files) > 5000 or \
                    any(os.path.exists(os.path.join(d, x, 'metadata.csv')) for x in dirs[:50]):
                dirs[:] = []
                continue
            pths += [os.path.join(d, f) for f in files if f == 'best.pth']
            zips += [os.path.join(d, f) for f in files if f.endswith('.zip')]
    return pths, zips


def find_checkpoints(roots):
    """Every runs/<name>/best.pth with its config.json; unpacks zipped notebook outputs if needed."""
    import zipfile
    found = {}
    pths, zips = _walk_inputs(roots)
    for p in pths:
        d = os.path.dirname(p)
        if os.path.exists(os.path.join(d, 'config.json')):
            found[os.path.basename(d)] = d
    for z in zips:
        try:
            with zipfile.ZipFile(z) as zf:
                names = [n for n in zf.namelist() if n.endswith(('best.pth', 'config.json', 'summary.json'))
                         and '/runs/' in '/' + n]
                for n in names:
                    run = os.path.basename(os.path.dirname(n))
                    if run in found:
                        continue
                    target = os.path.join('/tmp/ckpts', run)
                    os.makedirs(target, exist_ok=True)
                    if not os.path.exists(os.path.join(target, os.path.basename(n))):
                        with zf.open(n) as src, open(os.path.join(target, os.path.basename(n)), 'wb') as dst:
                            dst.write(src.read())
        except zipfile.BadZipFile:
            pass
    for d in glob.glob('/tmp/ckpts/*'):
        if os.path.exists(os.path.join(d, 'best.pth')) and os.path.exists(os.path.join(d, 'config.json')):
            found.setdefault(os.path.basename(d), d)
    return dict(sorted(found.items()))


def stage_score(a):
    import torch
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, Dataset
    from torchvision import transforms
    from cdg_train import Detector
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    crops = pd.read_csv(os.path.join(a.out, 'crops.csv'), keep_default_na=False)
    crops = crops[crops.ok == 1].reset_index(drop=True)
    tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])

    class DS(Dataset):
        def __init__(self, paths):
            self.paths = np.array(paths)

        def __len__(self):
            return len(self.paths)

        def __getitem__(self, i):
            return tf(Image.open(str(self.paths[i])).convert('RGB')), i

    ckpts = find_checkpoints(a.checkpoints)
    if a.only:
        ckpts = {k: v for k, v in ckpts.items() if any(k.startswith(p) for p in a.only)}
    log(f'{len(ckpts)} checkpoints: {", ".join(ckpts)}')
    if not ckpts:
        sys.exit('no checkpoints found: attach the E1 / E1b notebook outputs')
    out_path = os.path.join(a.out, 'scores.csv')
    done = set(pd.read_csv(out_path).run.unique()) if os.path.exists(out_path) else set()
    loader = DataLoader(DS(crops.crop.tolist()), batch_size=128, shuffle=False, num_workers=2)
    for run, d in ckpts.items():
        if run in done:
            continue
        cfg = json.load(open(os.path.join(d, 'config.json')))
        model = Detector(pretrained=False, spectral=cfg.get('spectral', 'band'),
                         channel_attn=cfg.get('channel_attn', 'off'), input_mode=cfg.get('input', 'dct'),
                         bands=cfg.get('bands', 16))
        model.load_state_dict(torch.load(os.path.join(d, 'best.pth'), map_location='cpu', weights_only=True))
        model = model.to(device).eval()
        probs = np.zeros(len(crops), dtype=np.float32)
        with torch.no_grad():
            for x, i in loader:
                x = x.to(device)
                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
                    logits = model(x, alpha=0.0)[0]
                p = F.softmax(logits.float(), 1)[:, 1]
                if not torch.isfinite(p).all():
                    p = F.softmax(model(x, alpha=0.0)[0].float(), 1)[:, 1]
                probs[i.numpy()] = p.cpu().numpy()
        thr = json.load(open(os.path.join(d, 'summary.json'))).get('threshold', 0.5) \
            if os.path.exists(os.path.join(d, 'summary.json')) else 0.5
        res = crops[['set', 'uid', 'view']].copy()
        res['run'], res['prob'], res['threshold'] = run, probs, thr
        res.to_csv(out_path, mode='a', header=not os.path.exists(out_path), index=False)
        log(f'scored {run}')


# ----------------------------------------------------------------------------
def stage_report(a):
    from sklearn.metrics import roc_auc_score
    from cdg_phase1 import md_table
    s = pd.read_csv(os.path.join(a.out, 'scores.csv'))
    rows = []
    for (run, view), g in s.groupby(['run', 'view']):
        real = g[g.set == 'Real']
        if real.empty:
            continue
        thr = g.threshold.iloc[0]
        rows.append(dict(run=run, view=view, fake_set='(reals flagged as fake)', n_real=len(real), n_fake=0,
                         AUC=np.nan, flagged=(real.prob >= thr).mean()))
        for fs, fg in g[g.set != 'Real'].groupby('set'):
            y = np.r_[np.zeros(len(real)), np.ones(len(fg))]
            p = np.r_[real.prob.values, fg.prob.values]
            rows.append(dict(run=run, view=view, fake_set=fs, n_real=len(real), n_fake=len(fg),
                             AUC=roc_auc_score(y, p), flagged=(fg.prob >= thr).mean()))
    r = pd.DataFrame(rows)
    parts = r.run.str.extract(r'^(?P<mode>[a-z]+)_(?P<sam>[a-z]+)_(?P<fold>[AB])_s(?P<seed>\d+)$')
    r = pd.concat([r, parts], axis=1)
    r['training_reals'] = r['mode'].map({'m': 'matched', 'o': 'original', 'band': 'original', 'se': 'original'})
    # E1 runs are named band_A_s1 / se_A_s1 (original reals)
    e1 = r.run.str.extract(r'^(?P<sam2>band|se)_(?P<fold2>[AB])_s(?P<seed2>\d+)$')
    r.loc[e1.sam2.notna(), 'sam'] = e1.sam2
    r.loc[e1.sam2.notna(), 'fold'] = e1.fold2
    r.loc[e1.sam2.notna(), 'seed'] = e1.seed2
    r.loc[e1.sam2.notna(), 'training_reals'] = 'original'
    fam = {k: v[3] for k, v in SETS.items()}
    r['family'] = r.fake_set.map(lambda x: af_family(x[3:]) if x.startswith('AF_') else fam.get(x, ''))
    r.to_csv(os.path.join(a.out, 'report_per_run.csv'), index=False)
    agg = (r[r.sam == 'band'].groupby(['view', 'training_reals', 'fold', 'family', 'fake_set'])
           .agg(n_seeds=('run', 'nunique'), AUC_mean=('AUC', 'mean'), AUC_sd=('AUC', 'std'),
                flagged_mean=('flagged', 'mean')).round(4).reset_index())
    agg.to_csv(os.path.join(a.out, 'report.csv'), index=False)
    c = pd.read_csv(os.path.join(a.out, 'crops.csv'), keep_default_na=False)
    c = c[c.view == 'asis']
    fmt = c.groupby('set').agg(n=('uid', 'count'), faces=('ok', 'sum'),
                               formats=('orig_format', lambda x: ', '.join(f'{k} {v}' for k, v in x.value_counts().items())),
                               median_w=('orig_w', 'median'), median_h=('orig_h', 'median'),
                               jpeg_q=('jpeg_q', lambda x: round(pd.to_numeric(x, errors='coerce').mean(), 1))).reset_index()
    md = [f'# External test: {a.title} (unseen generators)', '',
          'AUC of each generator against the same dataset\'s real faces. `asis` = files as distributed; `matched` = real and fake '
          'both re-saved as 512x512 JPEG quality 75 before cropping (no compression difference between classes). '
          '`flagged` = share scored as fake at the threshold chosen on the model\'s own validation set.', '',
          '## File formats per set (as distributed)', '', md_table(fmt), '',
          '## AUC, revised SAM, mean over seeds', '', md_table(agg)]
    open(os.path.join(a.out, 'report.md'), 'w').write('\n'.join(md))
    log('report written')
    return agg, fmt


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument('stage', choices=['fetch', 'artifact', 'crop', 'score', 'report'])
    ap.add_argument('--out', required=True)
    ap.add_argument('--sets', nargs='*', default=[], help='default: every sampled set')
    ap.add_argument('--n', type=int, default=1000, help='images kept per set')
    ap.add_argument('--stride', type=int, default=2, help='streaming: keep every stride-th image in archive order')
    ap.add_argument('--minutes', type=float, default=25, help='streaming: time limit per set')
    ap.add_argument('--min-keep', type=int, default=200, help='a set with fewer images counts as failed')
    ap.add_argument('--seed', type=int, default=2026)
    ap.add_argument('--local', default='', help='folder that already holds the .tar archives (skip download)')
    ap.add_argument('--artifact-root', nargs='+', default=['/kaggle/input'])
    ap.add_argument('--title', default='ArtiFact faces')
    ap.add_argument('--n-real', type=int, default=1500, help='ArtiFact: images per real face source')
    ap.add_argument('--checkpoints', nargs='+', default=['/kaggle/input'])
    ap.add_argument('--only', nargs='*', default=[], help='score only runs whose name starts with these')
    a = ap.parse_args(argv)
    if a.stage == 'fetch' and not a.sets:
        a.sets = list(SETS)
    os.makedirs(a.out, exist_ok=True)
    {'fetch': stage_fetch, 'artifact': stage_artifact, 'crop': stage_crop, 'score': stage_score, 'report': stage_report}[a.stage](a)


if __name__ == '__main__':
    main()

## 1. Inputs: ArtiFact and the checkpoints

In [ ]:
import sys; sys.path.insert(0, '.')
from cdg_external import find_checkpoints, find_artifact
metas = find_artifact(['/kaggle/input'])
print(len(metas), 'ArtiFact folders found', '(e.g. ' + os.path.dirname(metas[0]) + ')' if metas else '')
def show_tree(root='/kaggle/input', depth=4, per_dir=12):
    # what is attached: folder names to 4 levels (12 entries per folder) and every .csv near the top
    for d, dirs, files in os.walk(root, followlinks=True):
        lvl = d[len(root):].count(os.sep)
        if lvl >= depth:
            dirs[:] = []
        dirs.sort(); files.sort()
        print('  ' * lvl + os.path.basename(d) + '/', f'({len(dirs)} folders, {len(files)} files)',
              ', '.join([f for f in files if not f.lower().endswith(('.jpg', '.jpeg', '.png'))][:per_dir]))
        del dirs[per_dir:]
if not metas:
    show_tree()
assert metas, 'ArtiFact not found: the folder tree of the inputs is printed above (send it to Claude)'
ck = find_checkpoints(['/kaggle/input'])
ck = {k: v for k, v in ck.items() if k.startswith(('m_band', 'o_band', 'band_'))}   # the revised-SAM runs
print(len(ck), 'checkpoints:', ', '.join(ck))
assert ck, 'No checkpoints found: add the oct-22 and oct-23-1 notebook outputs as inputs'

## 2. Take a sample of the face images (copied from the attached dataset, nothing downloaded)

In [ ]:
sh(f'python cdg_external.py artifact --out {OUT} --artifact-root /kaggle/input --n 800 --n-real 1500')

## 3. Face crops (as distributed, and compression-matched)

In [ ]:
sh(f'python cdg_external.py crop --out {OUT}')

## 4. Score every checkpoint

In [ ]:
sh(f'python cdg_external.py score --out {OUT} --checkpoints /kaggle/input /tmp/ckpts --only m_band o_band band_')

## 5. Report

In [ ]:
sh(f'python cdg_external.py report --out {OUT} --title "ArtiFact faces"')
display(Markdown(open(f'{OUT}/report.md').read()))
shutil.rmtree(f'{OUT}/raw', ignore_errors=True)   # keep crops, scores and reports; drop the copied originals